In [14]:
!pip install streamlit pyngrok --quiet
!pip install transformers sentencepiece torch --quiet
!pip install elasticsearch pymilvus --quiet
!pip install spacy --quiet
!python -m spacy download en_core_web_lg
!pip install googletrans==4.0.0-rc1
!git clone https://github.com/vonhatbao2205/perception_models.git
%cd perception_models
!pip install -e .

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
datatrove 0.3.0 requires numpy<2.0.0,>=1.25.0, but you have numpy 2.3.4 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.3.4 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.15.3 which is incompatible.
mkl-umath 0.1.1 requires numpy<1.27.0,>=1.26.4, but you have numpy 2.3.4 which is incompatible.
mkl-random 1.2.4 requires numpy<1.27.0,>=1.26.4, but you have numpy 2.3.4 which is incompatible.
mkl-fft 1.3.8 requires numpy<1.27.0,>=1.26.4, but you have numpy 2.3.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.4 which is incompatible.
ydata-profiling 4.17.0 requires numpy<2.2,>=1.16.0, but you have numpy 2.3.4

In [2]:
!pip install -q open_clip_torch pillow tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.8 MB/s eta 0:00:00


In [ ]:
# Ver7

%%writefile app.py
# -*- coding: utf-8 -*-
import streamlit as st
import torch
import os
import json
import time
from PIL import Image, ImageDraw
import numpy as np
from pathlib import Path
import pandas as pd
import glob
import math
import requests
import io
import re

# Page config
try:
    st.set_page_config(layout="wide", page_title="🤖 ATLAS Retrieval")
except st.errors.StreamlitAPIException:
    pass

# Imports
try:
    import spacy
    from elasticsearch import Elasticsearch
    from pymilvus import connections, utility, Collection
    from kaggle_secrets import UserSecretsClient
    HYBRID_AVAILABLE = True
except ImportError:
    HYBRID_AVAILABLE = False
    UserSecretsClient = None

# MỚI: Thêm thư viện Dịch
try:
    import googletrans
    from googletrans import Translator
    TRANSLATE_AVAILABLE = True
except ImportError:
    TRANSLATE_AVAILABLE = False
    Translator = None

# Model imports
# Model imports: dùng CLIP ViT-H-14 từ open_clip
try:
    import open_clip
    CLIP_AVAILABLE = True
except ImportError:
    CLIP_AVAILABLE = False


# ==============================================================================
# CONFIGURATION
# ==============================================================================

CACHE_PATH = "/kaggle/input/image-path/image_paths_cache.json"
COLLECTION_NAME = "my_image_search_v2"
VECTOR_FIELD = "embedding_vector"
ID_FIELD = "image_path_id"
ES_ENDPOINT_URL = "https://my-elasticsearch-project-a0cb68.es.us-east-1.aws.elastic.cloud:443"
ES_INDEX_NAME = "objects"
ES_OBJECT_FIELD = "objects"
ES_IMAGE_ID_FIELD = "image_id"

# OCR Configuration
OCR_INDEX_NAME = "ocr_index"

# Transcript Configuration
TRANSCRIPT_INDEX_NAME = "transcripts_index"
YOUTUBE_MAP_PATH = '/kaggle/input/youtube-map/youtube_map.json'

# Paths
IMAGES_ROOTS = [
    "/kaggle/input/data-aic-2025-tnt",
    "/kaggle/input/aic-dataset-2025-s2"
]
MAP_KEYFRAMES_ROOTS = [
    "/kaggle/input/metadata-aic-2025/map-keyframes-aic25-b1/map-keyframes",
    "/kaggle/input/aic-dataset-2025-s2/map-keyframes-b2/map-keyframes-b2"
]
VIDEOS_ROOTS = ["/kaggle/input/video-aic-2025"]
MEDIA_INFO_ROOTS = [
    "/kaggle/input/metadata-aic-2025/media-info-aic25-b1/media-info",
    "/kaggle/input/aic-dataset-2025-s2/media-info-aic25-b2/media-info"
]

PERCEPTION_MODEL_NAME = "ViT-H-14"
# SỬA LỖI: Giới hạn topk của Milvus là 1024
MAX_FILTER_FETCH = 1024 
FILTER_FETCH_FACTOR = 10

# DRES API Configuration
DRES_BASE_URL = "https://eventretrieval.oj.io.vn"
DRES_USERNAME = "team004"
DRES_PASSWORD = "8PAmMMUts8"

# ==============================================================================
# DRES API FUNCTIONS
# ==============================================================================

def dres_login(username, password):
    """Logs into DRES and returns a session ID."""
    login_url = f"{DRES_BASE_URL}/api/v2/login"
    try:
        response = requests.post(login_url, json={"username": username, "password": password}, timeout=10)
        response.raise_for_status()  # Raise exception for bad status codes
        data = response.json()
        if "sessionId" in data:
            st.session_state.dres_session_id = data["sessionId"]
            st.sidebar.success(f"DRES Login OK. SessionID: ...{data['sessionId'][-6:]}")
        else:
            st.sidebar.error("DRES Login Failed: 'sessionId' not in response.")
    except requests.exceptions.RequestException as e:
        st.sidebar.error(f"DRES Login Error: {e}")

def dres_get_evaluation_id(session_id):
    """Gets the active evaluation ID."""
    if not session_id:
        st.sidebar.warning("Must login to DRES first.")
        return
    
    list_url = f"{DRES_BASE_URL}/api/v2/client/evaluation/list"
    try:
        response = requests.get(list_url, params={"session": session_id}, timeout=10)
        response.raise_for_status()
        evaluations = response.json()
        active_eval = None
        for eval in evaluations:
            if eval.get("status") == "ACTIVE":
                active_eval = eval
                break
        
        if active_eval:
            st.session_state.dres_evaluation_id = active_eval["id"]
            st.sidebar.success(f"Active Evaluation ID: {active_eval['id']}")
        else:
            st.sidebar.error("No 'ACTIVE' evaluation found.")
    except requests.exceptions.RequestException as e:
        st.sidebar.error(f"DRES Eval Error: {e}")

# === HÀM ĐƯỢC SỬA ===
def dres_submit(session_id, evaluation_id, payload):
    """Submits a payload to DRES and shows the result (CORRECT/WRONG)."""
    if not session_id or not evaluation_id:
        st.sidebar.error("Missing Session or Evaluation ID. Cannot submit.")
        return

    submit_url = f"{DRES_BASE_URL}/api/v2/submit/{evaluation_id}"
    try:
        response = requests.post(submit_url, params={"session": session_id}, json=payload, timeout=15)
        response.raise_for_status() # Sẽ báo lỗi nếu HTTP status là 4xx hoặc 5xx

        # Nếu code chạy đến đây, HTTP status là 200 (OK)
        data = response.json()
        
        api_status = data.get("status", "UNKNOWN_API_STATUS")
        # Đây là trường quan trọng:
        submission_result = data.get("submissionStatus", "UNKNOWN_RESULT") 

        # Hiển thị kết quả với màu sắc
        if submission_result == "CORRECT":
            st.sidebar.success(f"✅ NỘP BÀI THÀNH CÔNG: ĐÚNG (CORRECT)")
        elif submission_result == "PARTIALLY_CORRECT":
            st.sidebar.info(f"👍 NỘP BÀI THÀNH CÔNG: ĐÚNG MỘT PHẦN (PARTIALLY_CORRECT)")
        elif submission_result == "WRONG":
            st.sidebar.error(f"❌ NỘP BÀI THÀNH CÔNG: SAI (WRONG)")
        elif submission_result == "INDETERMINATE":
            st.sidebar.warning(f"🤔 NỘP BÀI THÀNH CÔNG: CHƯA XÁC ĐỊNH (INDETERMINATE)")
        else:
            # Fallback nếu DRES trả về status lạ
            st.sidebar.warning(f"⚠️ Nộp OK, nhưng kết quả không rõ: {submission_result} (API Status: {api_status})")
            st.sidebar.json(data) # In ra toàn bộ JSON để debug

    except requests.exceptions.RequestException as e:
        # Xử lý lỗi HTTP (ví dụ: 400, 401, 503)
        error_message = f"DRES Submit Error: {e}"
        if e.response is not None:
            try:
                # Cố gắng lấy thông báo lỗi chi tiết từ DRES
                error_data = e.response.json()
                error_detail = error_data.get('description', e.response.text)
                st.sidebar.error(f"Submit Failed ({e.response.status_code}): {error_detail}")
            except:
                # Nếu response không phải JSON
                st.sidebar.error(f"Submit Failed ({e.response.status_code}): {e.response.text}")
        else:
            st.sidebar.error(error_message)
    except Exception as e:
        # Xử lý các lỗi khác (ví dụ: lỗi parsing JSON)
        st.sidebar.error(f"Lỗi không xác định: {e}")
# === KẾT THÚC SỬA ===

# === MỚI: HÀM LẤY TASK HIỆN TẠI ===
def dres_get_current_task(session_id, evaluation_id):
    """
    Lấy thông tin task đang hoạt động (bao gồm câu hỏi/hints).
    """
    if not session_id or not evaluation_id:
        st.sidebar.error("Chưa login hoặc chưa có Evaluation ID.")
        return None
    
    # Đây là endpoint API chuẩn của DRES mà client dùng để lấy task
    task_url = f"{DRES_BASE_URL}/api/v2/client/evaluation/currentTask"
    
    try:
        response = requests.get(task_url, params={"session": session_id, "evaluationId": evaluation_id}, timeout=10)
        response.raise_for_status() # Báo lỗi nếu status code là 4xx/5xx
        
        task_data = response.json()
        
        # task_data sẽ có dạng:
        # { "taskName": "tkis-01", "taskType": "KIS", "remainingTime": 290000,
        #   "hints": [
        #     { "type": "TEXT", "content": "Một con robot có 4 chân...", "startTime": 0 },
        #     { "type": "VIDEO", "content": "path/to/video.mp4", "startTime": 30000 }
        #   ]
        # }
        
        return task_data
        
    except requests.exceptions.RequestException as e:
        # Xử lý lỗi (ví dụ: task chưa bắt đầu, hoặc đã kết thúc)
        try:
            error_detail = e.response.json().get('description', e.response.text) if e.response else str(e)
            st.sidebar.error(f"Lỗi khi tải task: {error_detail}")
        except: # Fallback nếu response không phải là JSON
             st.sidebar.error(f"Lỗi khi tải task: {e}")
        return None
# === KẾT THÚC HÀM MỚI ===

# ==============================================================================
# METADATA SYSTEM
# ==============================================================================

class MetadataSystem:
    def __init__(self, images_roots, map_keyframes_roots, videos_roots, media_info_roots):
        self.images_roots = [Path(p) for p in images_roots]
        self.map_keyframes_roots = [Path(p) for p in map_keyframes_roots]
        self.videos_roots = [Path(p) for p in videos_roots]
        self.media_info_roots = [Path(p) for p in media_info_roots]

        self.rel_to_full_path_map = self._load_or_build_path_map()
        self.master_relative_paths = sorted(list(self.rel_to_full_path_map.keys()))
        self.frame_idx_map, self.video_fps_map, self.pts_time_map = self._load_frame_idx_and_fps_maps(self.map_keyframes_roots)
        self.youtube_url_map = self._load_media_info(self.media_info_roots)
        self.video_to_sorted_frames = self._build_sorted_frame_map()
        self.video_path_map = self._map_video_files()

    def _load_or_build_path_map(self):
        if os.path.exists(CACHE_PATH):
            try:
                with open(CACHE_PATH, 'r') as f:
                    return json.load(f)
            except:
                pass
        
        path_map = {}
        for root_path in self.images_roots:
            if os.path.exists(root_path):
                prefix = root_path.name
                for dirpath, _, filenames in os.walk(root_path):
                    for filename in filenames:
                        if filename.lower().endswith(".jpg"):
                            full_path_str = os.path.join(dirpath, filename)
                            rel_path = str(Path(full_path_str).relative_to(root_path))
                            path_map[rel_path] = full_path_str
                            prefixed_path = f"{prefix}/{rel_path}"
                            path_map[prefixed_path] = full_path_str
        
        try:
            os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
            with open(CACHE_PATH, 'w') as f:
                json.dump(path_map, f)
        except:
            pass
            
        return path_map

    def get_full_path(self, rel_path):
        full_path_str = self.rel_to_full_path_map.get(rel_path)
        return Path(full_path_str) if full_path_str else None

    def _format_time(self, seconds):
        if pd.isna(seconds) or seconds < 0:
            return "N/A"
        minutes = int(seconds // 60)
        secs = int(seconds % 60)
        return f"{minutes}:{secs:02d}"

    def _load_frame_idx_and_fps_maps(self, map_roots):
        idx_mapping, fps_mapping, pts_time_mapping = {}, {}, {}
        all_csv_files = []
        for root in map_roots:
            if root.exists():
                all_csv_files.extend(root.glob('**/*.csv'))
                
        for csv_file in all_csv_files:
            try:
                video_id = csv_file.stem
                df = pd.read_csv(csv_file)
                df.columns = [c.strip().replace('#', '') for c in df.columns]
                if 'n' in df.columns and 'frame_idx' in df.columns and 'fps' in df.columns:
                    df.dropna(subset=['n', 'frame_idx', 'fps'], inplace=True)
                    idx_mapping[video_id] = pd.to_numeric(df.set_index('n')['frame_idx'], errors='coerce').to_dict()
                    if not df.empty:
                        fps_mapping[video_id] = float(df['fps'].iloc[0])
                    if 'pts_time' in df.columns:
                        pts_time_mapping[video_id] = pd.to_numeric(df.set_index('n')['pts_time'], errors='coerce').to_dict()
            except:
                continue
        return idx_mapping, fps_mapping, pts_time_mapping

    def _load_media_info(self, media_info_roots):
        url_map = {}
        all_json_files = []
        for root in media_info_roots:
            if root.exists():
                all_json_files.extend(root.glob('**/*.json'))
        for json_file in all_json_files:
            try:
                video_id = json_file.stem
                with open(json_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                if 'watch_url' in data:
                    url_map[video_id] = data['watch_url']
            except:
                continue
        return url_map

    def _map_video_files(self):
        video_map = {}
        video_files = []
        for root in self.videos_roots:
            if os.path.exists(root):
                video_files.extend(glob.glob(os.path.join(root, "**", "*.mp4"), recursive=True))
        for f in video_files:
            video_map[Path(f).stem] = f
        return video_map

    def _get_structure_paths(self):
        dataset_names = [root.name for root in self.images_roots]
        return [p for p in self.master_relative_paths if Path(p).parts[0] not in dataset_names]

    def _build_sorted_frame_map(self):
        video_map, sorted_video_map = {}, {}
        for p in self._get_structure_paths():
            try:
                parts = Path(p).parts
                if len(parts) >= 3:
                    video_id = parts[2]
                    video_map.setdefault(video_id, []).append(p)
            except:
                continue
        
        for vid, paths in video_map.items():
            frames = []
            for p in paths:
                try:
                    frame_num = int(Path(p).stem)
                    idx = self.frame_idx_map.get(vid, {}).get(frame_num, -1)
                    if idx != -1:
                        frames.append((idx, p))
                except:
                    continue
            sorted_video_map[vid] = sorted(frames, key=lambda x: x[0])
        return sorted_video_map

    def get_folder_structure(self):
        structure = {}
        for path in self._get_structure_paths():
            parts = Path(path).parts
            if len(parts) >= 3:
                kf = parts[0]
                vid = parts[2]
                structure.setdefault(kf, []).append(vid)
        for kf in structure:
            structure[kf] = sorted(list(set(structure[kf])))
        return structure

    def get_image_info(self, rel_path):
        try:
            path_parts = Path(rel_path).parts
            kf_idx, vid_idx = 0, 2
            dataset_names = [root.name for root in self.images_roots]
            if path_parts[0] in dataset_names:
                kf_idx, vid_idx = 1, 3

            if len(path_parts) > vid_idx:
                keyframe_folder = path_parts[kf_idx]
                video_id = path_parts[vid_idx]
                frame_num_str = Path(rel_path).stem
                try:
                    frame_num = int(frame_num_str)
                except ValueError:
                    frame_num = -1

                frame_idx = self.frame_idx_map.get(video_id, {}).get(frame_num, "N/A")
                pts_time_seconds = self.pts_time_map.get(video_id, {}).get(frame_num, None)
                pts_time_formatted = self._format_time(pts_time_seconds)
                return {
                    "keyframe_folder": keyframe_folder,
                    "folder_name": video_id,
                    "frame_index": str(frame_idx),
                    "image_name": frame_num_str,
                    "pts_time": pts_time_formatted,
                    "pts_time_seconds": pts_time_seconds if pts_time_seconds is not None else -1 # Thay đổi: trả về -1 nếu None
                }
        except:
            pass
        return {
            "keyframe_folder": "Unknown",
            "folder_name": "Unknown",
            "frame_index": "N/A",
            "image_name": Path(rel_path).stem,
            "pts_time": "N/A",
            "pts_time_seconds": -1 # Thay đổi: trả về -1 nếu None
        }

    # === HÀM ĐƯỢC SỬA ===
    def calculate_metadata(self, video_id, image_filename):
        """Calculate frame index, timestamp (ms), and video URL"""
        try:
            match = re.search(r'(\d+)\.jpg$', image_filename, re.IGNORECASE)
            if match:
                keyframe_n = int(match.group(1))
            else:
                raise ValueError("Invalid filename format")
        except:
            # Sửa: Trả về 8 giá trị
            return "Format index: Error", "Format time: Error", None, None, 0, 0, "ERROR", "ERROR"

        try:
            frame_idx = self.frame_idx_map[video_id][keyframe_n]
        except KeyError:
            # Sửa: Trả về 8 giá trị
            return f"Format index: {video_id}-NotFound", f"Format time: {video_id}-NotFound", None, None, 0, 0, "NOT_FOUND", "NOT_FOUND"

        fps = self.video_fps_map.get(video_id, 25.0)
        time_ms = 0
        time_sec = 0
        
        try:
            if fps > 0:
                time_sec_float = frame_idx / fps
                time_ms = int(time_sec_float * 1000)
                time_sec = math.floor(time_sec_float)
        except:
            # Sửa: Trả về 8 giá trị
            format_index = f"{video_id}-{frame_idx}"
            return f"Format index: {format_index}", "Format time: Error", None, None, 0, frame_idx, format_index, "ERROR"

        base_url = self.youtube_url_map.get(video_id)
        format_index = f"{video_id}-{frame_idx}"
        format_time = f"{video_id}-{time_ms}(ms)"
        format_time_copyable = f"{video_id}-{time_ms}" # MỚI

        # Sửa: Trả về 8 giá trị
        return f"Idx: {format_index}", f"Time: {format_time}", base_url, time_sec, time_ms, frame_idx, format_index, format_time_copyable
    # === KẾT THÚC SỬA ===

    def search_neighbors(self, frame_id, num_neighbors):
        try:
            video_id, fnum_str = frame_id.replace(" ", "").split('/')
        except:
            return None, "Invalid Frame ID format"
        
        sorted_frames = self.video_to_sorted_frames.get(video_id)
        if not sorted_frames:
            return None, f"Video ID not found: {video_id}"
        
        t_idx = -1
        for i, (_, p) in enumerate(sorted_frames):
            if Path(p).stem == fnum_str:
                t_idx = i
                break

        if t_idx == -1:
            return None, f"Frame '{fnum_str}' not found"
        
        s = max(0, t_idx - num_neighbors)
        e = min(len(sorted_frames), t_idx + num_neighbors + 1)
        return sorted_frames[s:e], None

# ==============================================================================
# CACHING FUNCTIONS
# ==============================================================================

# MỚI: Hàm Dịch (có cache và xử lý lỗi)
@st.cache_data
def translate_to_english(text_query):
    """
    Dịch văn bản sang Tiếng Anh, sử dụng cache và xử lý lỗi an toàn.
    """
    if not TRANSLATE_AVAILABLE:
        st.sidebar.warning("Thư viện googletrans chưa được cài đặt. Bỏ qua dịch.")
        return text_query
    if not text_query or not text_query.strip():
        return text_query
        
    try:
        # Khởi tạo Translator() bên trong hàm
        # (Đối tượng Translator không thể cache an toàn)
        translator = Translator()
        
        # Kiểm tra ngôn ngữ
        detected_lang = translator.detect(text_query).lang
        if detected_lang.lower().startswith('en'):
            return text_query # Đã là Tiếng Anh
        
        # Dịch
        translated = translator.translate(text_query, dest='en')
        if translated and translated.text:
            return translated.text
        return text_query # Fallback
    except Exception as e:
        st.sidebar.warning(f"Lỗi API Dịch: {e}. Sử dụng văn bản gốc.")
        return text_query


@st.cache_resource
def get_secrets():
    if not HYBRID_AVAILABLE or UserSecretsClient is None:
        return {}
    try:
        user_secrets = UserSecretsClient()
        return {
            "MILVUS_API_KEY": user_secrets.get_secret("MILVUS_API_KEY"),
            "MILVUS_ENDPOINT": user_secrets.get_secret("MILVUS_ENDPOINT"),
            "ES_API_KEY": user_secrets.get_secret("ES_API_KEY")
        }
    except Exception as e:
        print(f"Error loading secrets: {e}")
        return {}

@st.cache_resource
def load_perception_model(model_name):
    """
    Thực ra giờ load CLIP ViT-H-14 (open_clip) thay cho PE.
    model_name = "ViT-H-14"
    pretrained = "laion2b_s32b_b79k"  (LAION-2B)
    """
    if not CLIP_AVAILABLE:
        return None, None, None, None
    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        # tạo model + transform
        model, _, preprocess = open_clip.create_model_and_transforms(
            model_name,
            pretrained="laion2b_s32b_b79k",
        )
        model = model.to(device).eval()

        # tokenizer cho text
        tokenizer = open_clip.get_tokenizer(model_name)

        image_transform = preprocess
        return model, tokenizer, device, image_transform
    except Exception as e:
        print(f"Error loading CLIP model: {e}")
        return None, None, None, None


@st.cache_resource
def connect_to_milvus(endpoint, api_key, collection_name):
    if not endpoint or not api_key:
        return None
    try:
        connections.connect("default", uri=endpoint, token=api_key)
        if not utility.has_collection(collection_name):
            return None
        collection = Collection(collection_name)
        collection.load()
        return collection
    except Exception as e:
        print(f"Milvus connection error: {e}")
        return None

@st.cache_resource
def get_es_client(endpoint, api_key):
    if not endpoint or not api_key:
        return None
    try:
        es = Elasticsearch(endpoint, api_key=api_key)
        if es.ping():
            return es
        return None
    except Exception as e:
        print(f"ES connection error: {e}")
        return None

@st.cache_resource
def get_spacy_model():
    try:
        return spacy.load("en_core_web_lg")
    except:
        return None

@st.cache_resource
def load_metadata_system():
    return MetadataSystem(IMAGES_ROOTS, MAP_KEYFRAMES_ROOTS, VIDEOS_ROOTS, MEDIA_INFO_ROOTS)

# TÍNH NĂNG MỚI: Cache hàm get_image_info để tăng tốc Trake Search
@st.cache_data
def get_image_info_cached(_system, rel_path):
    # Hàm _system.get_image_info không an toàn cho cache vì _system là object
    # Thay vào đó, chúng ta sẽ gọi trực tiếp
    return load_metadata_system().get_image_info(rel_path)

@st.cache_data
def get_all_tags(_es_client):
    if not _es_client:
        return []
    try:
        query_body = {
            "size": 0,
            "aggs": {
                "all_tags": {
                    "terms": {
                        "field": ES_OBJECT_FIELD,
                        "size": 1500
                    }
                }
            }
        }
        response = _es_client.search(index=ES_INDEX_NAME, body=query_body)
        tags = [bucket['key'] for bucket in response['aggregations']['all_tags']['buckets']]
        tags.sort()
        return tags
    except:
        return []

@st.cache_data
def get_cached_folder_structure(_system):
    return _system.get_folder_structure()

@st.cache_data
def get_all_collections(_es_client):
    if not _es_client:
        return []
    try:
        body = {
            "size": 0,
            "aggs": {
                "collections": {
                    "terms": {
                        "field": "collection_id",
                        "size": 1000
                    }
                }
            }
        }
        response = _es_client.search(index=OCR_INDEX_NAME, body=body)
        collections = [bucket['key'] for bucket in response['aggregations']['collections']['buckets'] 
                      if bucket['key'] is not None]
        collections.sort()
        return collections
    except:
        return []

@st.cache_data
def get_all_video_ids(_es_client):
    if not _es_client:
        return []
    try:
        body = {
            "size": 0,
            "aggs": {
                "videos": {
                    "terms": {
                        "field": "video_id",
                        "size": 5000
                    }
                }
            }
        }
        response = _es_client.search(index=TRANSCRIPT_INDEX_NAME, body=body)
        videos = [bucket['key'] for bucket in response['aggregations']['videos']['buckets'] 
                 if bucket['key'] is not None]
        videos.sort()
        return videos
    except:
        return []

@st.cache_data
def load_video_metadata():
    if not os.path.exists(YOUTUBE_MAP_PATH):
        return None
    try:
        with open(YOUTUBE_MAP_PATH, 'r', encoding='utf-8') as f:
            return json.load(f)
    except:
        return None

# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================

def parse_english_text(en_text, nlp_model):
    if not en_text or not nlp_model:
        return []
    doc = nlp_model(en_text)
    objects = [token.text.lower() for token in doc if token.tag_ in ['NN', 'NNS']]
    common_words = ['photo', 'image', 'picture', 'person', 'video', 'frame', 'people']
    return list(set([obj for obj in objects if obj not in common_words]))

def format_time(seconds):
    if seconds is None:
        return "00:00"
    return time.strftime('%M:%S', time.gmtime(float(seconds)))

# === HÀM ĐƯỢC SỬA ===
def display_gallery_with_video(results, system, key_prefix, cols_per_row=4, 
                               highlight_target_id=None, show_video_player=True):
    """Enhanced gallery with options to show/hide video"""
    if not results:
        return

    images_data = []
    
    target_video_id, target_frame_name = None, None
    if highlight_target_id:
        try:
            target_video_id, target_frame_name = highlight_target_id.split('/')
        except:
            pass

    for res in results:
        if isinstance(res, tuple):
            frame_idx_from_tuple, rel_path = res
            score_info = {}
        else:
            rel_path = res["path"]
            score_info = res
            frame_idx_from_tuple = None

        full_path = system.get_full_path(rel_path)
        
        if full_path and full_path.exists():
            try:
                img_info = system.get_image_info(rel_path)
                video_id = img_info['folder_name']
                frame_name = img_info['image_name']
                
                frame_id = f"{video_id}/{frame_name}"

                # Calculate metadata
                # Sửa: Nhận 8 giá trị
                format_index_str, format_time_str, base_video_url, start_time_sec, time_ms, frame_idx, copy_idx_str, copy_time_str = system.calculate_metadata(
                    video_id, f"{frame_name}.jpg"
                )

                # Use frame_idx from tuple if available (for neighbor search)
                current_frame_idx = frame_idx_from_tuple if frame_idx_from_tuple is not None else frame_idx

                # Load and process image
                img = Image.open(str(full_path)).convert("RGB")
                
                # Highlight if target
                is_target = (video_id == target_video_id and frame_name == target_frame_name)
                if is_target:
                    draw = ImageDraw.Draw(img)
                    draw.rectangle([(0,0), (img.width-1, img.height-1)], outline="yellow", width=20)

                # Convert to bytes
                buf = io.BytesIO()
                img.save(buf, format="JPEG")
                byte_im = buf.getvalue()

                images_data.append({
                    'bytes': byte_im,
                    'frame_id': frame_id,
                    'video_id': video_id,
                    'frame_idx': current_frame_idx,
                    'time_ms': time_ms,
                    'format_index': format_index_str, # "Idx: K14_V018-13657"
                    'format_time': format_time_str,   # "Time: K14_V018-546280(ms)"
                    'video_url': base_video_url,
                    'start_time': start_time_sec,
                    'is_target': is_target,
                    'score': score_info.get('score'),
                    'copy_idx_str': copy_idx_str,     # "K14_V018-13657"
                    'copy_time_str': copy_time_str   # "K14_V018-546280"
                })

            except:
                pass

    if not images_data:
        return

    st.write(f"📊 Displaying {len(images_data)} results:")
    cols = st.columns(cols_per_row)
    
    for i, data in enumerate(images_data):
        with cols[i % cols_per_row]:
            # Display image
            if data['is_target']:
                st.markdown("### ⭐ TARGET")
            
            st.image(data['bytes'], use_container_width=True)
            
            # === SỬA: Hiển thị Idx/Time với st.code ===
            st.markdown(f"**Idx:**")
            st.code(data['copy_idx_str'], language=None) # Ngôn ngữ None để không highlight
            
            st.markdown(f"**Time (ms):**")
            st.code(data['copy_time_str'], language=None)
            # === KẾT THÚC SỬA ===
            
            if data['score'] is not None:
                st.caption(f"Score: {data['score']:.5f}")
            
            # Video player in expander (conditional)
            if show_video_player and data['video_url'] and data['start_time'] is not None:
                with st.expander("▶️ Watch Video"):
                    st.video(data['video_url'], start_time=data['start_time'])
            
            # Select button for neighbor search
            if st.button("Select", key=f"{key_prefix}_btn_{i}"):
                st.session_state.selected_frame_id = data['frame_id']
                st.rerun()
# === KẾT THÚC SỬA HÀM ===


# ==============================================================================
# SEARCH FUNCTIONS
# ==============================================================================

def encode_text_query(text_query, model, tokenizer, device):
    if not model or not tokenizer or not text_query:
        return []
    try:
        with torch.no_grad():
            text_tokens = tokenizer([text_query]).to(device)
            with torch.autocast(device.type, enabled=device.type != 'cpu'):
                text_features = model.encode_text(text_tokens)
            text_features /= text_features.norm(dim=-1, keepdim=True)
            return text_features.cpu().numpy().tolist()
    except Exception as e:
        st.error(f"Text encoding error: {e}")
        return []


def encode_image_query(image, model, image_transforms, device):
    if not model or not image_transforms:
        return []
    if image.mode != 'RGB':
        image = image.convert('RGB')
    try:
        image_tensor = image_transforms(image).unsqueeze(0).to(device)
        with torch.no_grad():
            with torch.autocast(device.type, enabled=device.type != 'cpu'):
                image_features = model.encode_image(image_tensor)
            image_features /= image_features.norm(dim=-1, keepdim=True)
            return image_features.cpu().numpy().tolist()
    except Exception as e:
        st.error(f"Image encoding error: {e}")
        return []


def apply_post_filtering(ranked_list, kf_filter, vid_filter, system, top_k):
    if kf_filter == "All":
        return ranked_list[:top_k]

    filtered_list = []
    for item in ranked_list:
        path = item['path']
        info = system.get_image_info(path)
        
        if info['keyframe_folder'] == kf_filter:
            if vid_filter == "All" or info['folder_name'] == vid_filter:
                filtered_list.append(item)
                if len(filtered_list) >= top_k:
                    break
    return filtered_list

def search_milvus_by_vector(collection, query_vector, top_k, kf_filter, vid_filter, system):
    if not collection or not query_vector:
        return []
    
    fetch_k = top_k
    if kf_filter != "All":
        fetch_k = min(MAX_FILTER_FETCH, top_k * FILTER_FETCH_FACTOR)

    search_params = {"metric_type": "COSINE", "params": {}}
    try:
        results = collection.search(
            data=query_vector,
            anns_field=VECTOR_FIELD,
            param=search_params,
            limit=fetch_k,
            output_fields=[ID_FIELD]
        )
        
        ranked_list = []
        for hit in results[0]:
            similarity = 1.0 - max(0.0, hit.distance)
            ranked_list.append({"path": hit.id, "score": similarity})
        
        # SỬA: Chuyển 'system' sang 'get_image_info_cached'
        # ranked_list sẽ được lọc sau khi có thông tin
        
        # Lấy thông tin cho tất cả các kết quả
        results_with_info = []
        for item in ranked_list:
            info = get_image_info_cached(system, item['path']) # Dùng hàm cache
            item['info'] = info
            results_with_info.append(item)

        # Lọc thủ công (thay thế apply_post_filtering)
        if kf_filter == "All":
            return results_with_info[:top_k]
        
        filtered_list = []
        for item in results_with_info:
            if item['info']['keyframe_folder'] == kf_filter:
                if vid_filter == "All" or item['info']['folder_name'] == vid_filter:
                    filtered_list.append(item)
                    if len(filtered_list) >= top_k:
                        break
        return filtered_list

    except Exception as e:
        st.error(f"Milvus search error: {e}")
        return []

# TÍNH NĂNG MỚI: Hàm tìm kiếm Milvus được tối ưu hóa cho Trake
def search_milvus_for_trake(collection, query_vector, top_k, system):
    """
    Tìm kiếm Milvus và trả về kết quả kèm thông tin (info) đã được cache.
    Không thực hiện post-filtering.
    """
    if not collection or not query_vector:
        return []

    search_params = {"metric_type": "COSINE", "params": {}}
    try:
        results = collection.search(
            data=query_vector,
            anns_field=VECTOR_FIELD,
            param=search_params,
            limit=top_k, # Lấy chính xác top_k
            output_fields=[ID_FIELD]
        )
        
        ranked_list = []
        for hit in results[0]:
            similarity = 1.0 - max(0.0, hit.distance)
            item = {"path": hit.id, "score": similarity}
            # Lấy thông tin từ cache ngay lập tức
            item['info'] = get_image_info_cached(system, item['path'])
            ranked_list.append(item)
        
        return ranked_list
    except Exception as e:
        st.error(f"Milvus (Trake) search error: {e}")
        return []


def search_elasticsearch(es_client, objects_list, top_k, kf_filter, vid_filter, system, match_mode='should'):
    if not es_client or not objects_list:
        return []

    fetch_k = top_k
    if kf_filter != "All":
        fetch_k = min(MAX_FILTER_FETCH, top_k * FILTER_FETCH_FACTOR)

    try:
        term_queries = [{"match": {ES_OBJECT_FIELD: tag}} for tag in objects_list]
        if match_mode == 'must':
            query_body = {"size": fetch_k, "query": {"bool": {"must": term_queries}}}
        else:
            query_body = {"size": fetch_k, "query": {"bool": {"should": term_queries}}}
        
        response = es_client.search(index=ES_INDEX_NAME, body=query_body)
        
        ranked_list = []
        for hit in response['hits']['hits']:
            try:
                image_id = hit['_source'][ES_IMAGE_ID_FIELD]
            except KeyError:
                continue
            score = hit['_score'] if match_mode == 'should' else 1.0
            ranked_list.append({"path": image_id, "score": score})
        
        # SỬA: Chuyển 'system' sang 'get_image_info_cached'
        # ranked_list sẽ được lọc sau khi có thông tin
        
        # Lấy thông tin cho tất cả các kết quả
        results_with_info = []
        for item in ranked_list:
            info = get_image_info_cached(system, item['path']) # Dùng hàm cache
            item['info'] = info
            results_with_info.append(item)

        # Lọc thủ công (thay thế apply_post_filtering)
        if kf_filter == "All":
            return results_with_info[:top_k]
        
        filtered_list = []
        for item in results_with_info:
            if item['info']['keyframe_folder'] == kf_filter:
                if vid_filter == "All" or item['info']['folder_name'] == vid_filter:
                    filtered_list.append(item)
                    if len(filtered_list) >= top_k:
                        break
        return filtered_list

    except Exception as e:
        st.error(f"ES search error: {e}")
        return []

def rank_wrrf(results_list, weights, k=60):
    if not results_list:
        return []
    if len(results_list) != len(weights):
        weights = [1.0] * len(results_list)

    fused_scores = {}
    original_scores = {}

    for list_index, (results, weight) in enumerate(zip(results_list, weights)):
        for rank, item in enumerate(results, start=1):
            path = item['path']
            rrf_score = weight * (1.0 / (k + rank))
            fused_scores[path] = fused_scores.get(path, 0.0) + rrf_score
            
            if path not in original_scores:
                original_scores[path] = {}
            source_name = "vector" if list_index == 0 else "object"
            original_scores[path][f'{source_name}_score'] = item.get('score', 0.0)
            
            # TÍNH NĂNG MỚI: Mang theo 'info' nếu có
            if 'info' in item:
                original_scores[path]['info'] = item['info']

    final_results = []
    for path, score in fused_scores.items():
        result_item = {"path": path, "score": score}
        result_item.update(original_scores.get(path, {}))
        final_results.append(result_item)

    final_results.sort(key=lambda x: x["score"], reverse=True)
    return final_results

# TÍNH NĂNG MỚI: Hàm logic chính cho Trake Search (Giai đoạn 2)
def find_temporal_sequences(event_results_list, max_sequences, kf_filter="All", vid_filter="All"): # THÊM kf_filter, vid_filter
    """
    Tìm các chuỗi hợp lệ từ danh sách kết quả của các sự kiện.
    event_results_list: List[List[result_dict]], trong đó result_dict đã chứa 'info'.
    """
    valid_sequences = []
    
    if not event_results_list:
        return []
    
    num_events = len(event_results_list)
    
    # 1. Nhóm kết quả theo video_id cho từng sự kiện
    video_event_maps = []
    all_common_video_ids = set()
    
    for i, event_results in enumerate(event_results_list):
        video_map = {}
        video_ids_for_this_event = set()
        for res in event_results:
            video_id = res['info'].get('folder_name')
            keyframe_folder = res['info'].get('keyframe_folder') # MỚI: Lấy keyframe folder
            
            if video_id == "Unknown" or res['info'].get('pts_time_seconds', -1) < 0:
                continue
            
            # === LỌC LOGIC MỚI ===
            if kf_filter != "All" and keyframe_folder != kf_filter:
                continue # Bỏ qua nếu không đúng keyframe folder
            
            if vid_filter != "All" and video_id != vid_filter:
                continue # Bỏ qua nếu không đúng video
            # === KẾT THÚC LỌC LOGIC MỚI ===

            video_map.setdefault(video_id, []).append(res)
            video_ids_for_this_event.add(video_id)
            
        video_event_maps.append(video_map)
        
        if i == 0:
            all_common_video_ids = video_ids_for_this_event
        else:
            all_common_video_ids.intersection_update(video_ids_for_this_event)

    if not all_common_video_ids:
        return []

    # 2. Thực hiện "join" tuần tự
    for video_id in all_common_video_ids:
        # Bắt đầu với các khung hình của sự kiện đầu tiên
        sequences_in_progress = [ [res] for res in video_event_maps[0].get(video_id, []) ]
        
        # Lặp qua các sự kiện còn lại (E2, E3, ...)
        for i in range(1, num_events):
            new_sequences_in_progress = []
            event_frames = video_event_maps[i].get(video_id, [])
            
            if not event_frames: # Nếu video này không có kết quả cho sự kiện i
                sequences_in_progress = [] # Xóa tất cả chuỗi đang xử lý
                break # Dừng xử lý video này
            
            for seq in sequences_in_progress:
                last_event_frame = seq[-1]
                last_timestamp = last_event_frame['info']['pts_time_seconds']
                
                # Tìm tất cả các khung hình E[i] xảy ra SAU E[i-1]
                for current_event_frame in event_frames:
                    current_timestamp = current_event_frame['info']['pts_time_seconds']
                    
                    if current_timestamp > last_timestamp:
                        # Tìm thấy một cặp hợp lệ, tạo chuỗi mới
                        new_seq = seq + [current_event_frame]
                        new_sequences_in_progress.append(new_seq)
            
            sequences_in_progress = new_sequences_in_progress
            if not sequences_in_progress:
                break # Không tìm thấy cặp E[i-1] < E[i] nào, dừng video này

        # Sau khi lặp qua tất cả sự kiện
        valid_sequences.extend(sequences_in_progress)
        
        if len(valid_sequences) >= max_sequences:
            return valid_sequences[:max_sequences]

    # TODO: Sắp xếp 'valid_sequences' dựa trên tổng điểm (score)
    # Tạm thời chỉ trả về theo thứ tự tìm thấy
    return valid_sequences

# ==============================================================================
# MAIN UI
# ==============================================================================

def initialize_state():
    defaults = {
        'search_results': [],
        'image_search_results': [],
        'selected_frame_id': None,
        'neighbor_results': [],
        'ocr_results': [],
        'transcript_results': [],
        'nlp_extracted_objects': [],
        'dres_session_id': None,
        'dres_evaluation_id': None,
        'kis_input': '',
        'qa_answer': '',
        'qa_input': '',
        'trake_input': '',
        'trake_sequences_results': [], # TÍNH NĂNG MỚI: Lưu kết quả Trake
        'current_task_data': None # MỚI: Lưu trữ task hiện tại
    }
    for key, value in defaults.items():
        if key not in st.session_state:
            st.session_state[key] = value

def main_ui():
    st.title("🤖 ATLAS Retrieval")
    
    initialize_state()
    
    # Load resources
    system = load_metadata_system()
    model, tokenizer, device, image_transform = load_perception_model(PERCEPTION_MODEL_NAME)
    
    secrets = get_secrets()
    milvus_collection = None
    es_client = None
    nlp_model = None
    all_object_tags = []
    
    if secrets:
        milvus_collection = connect_to_milvus(
            secrets.get("MILVUS_ENDPOINT"),
            secrets.get("MILVUS_API_KEY"),
            COLLECTION_NAME
        )
        es_client = get_es_client(ES_ENDPOINT_URL, secrets.get("ES_API_KEY"))
        nlp_model = get_spacy_model()
    
    folder_structure = get_cached_folder_structure(system)
    available_keyframes = sorted(list(folder_structure.keys()))
    filter_keyframes = ["All"] + available_keyframes
    
    # Sidebar
    with st.sidebar:
        st.markdown("## ⚙️ Settings")
        
        if st.button("🆕 New Search / Clear All", use_container_width=True):
            # Keep DRES credentials
            session_id = st.session_state.dres_session_id
            eval_id = st.session_state.dres_evaluation_id
            
            # Clear all
            for key in st.session_state.keys():
                del st.session_state[key]
            
            # Re-initialize and restore DRES
            initialize_state()
            st.session_state.dres_session_id = session_id
            st.session_state.dres_evaluation_id = eval_id
            
            st.rerun()
        
        st.markdown("### 📂 Filters")
        st.selectbox("Keyframes Folder", options=filter_keyframes, key="kf_filter")
        
        if st.session_state.kf_filter == "All":
            video_choices = ["All"]
        else:
            video_choices = ["All"] + folder_structure.get(st.session_state.kf_filter, [])
        
        current_vid = st.session_state.get("vid_filter", "All")
        if current_vid not in video_choices:
            st.session_state.vid_filter = "All"
        
        st.selectbox("Video Folder", options=video_choices, key="vid_filter")
        
        # MỚI: Thêm nút dịch
        st.checkbox(
            "Dịch truy vấn sang Tiếng Anh", 
            value=True, # Mặc định là có
            key="translate_to_en",
            help="Tự động dịch văn bản (Tiếng Việt,...) sang Tiếng Anh trước khi tìm kiếm."
        )

        st.slider("Top K Results", 1, 200, 12, 1, key="top_k")
        st.slider("Số cột hiển thị", 1, 10, 4, 1, key="cols_per_row")
        with st.expander("WRRF Settings"):
            st.number_input("Vector Weight", 0.0, value=1.0, step=0.1, key="w_vector")
            st.number_input("Object Weight", 0.0, value=1.0, step=0.1, key="w_object")
            st.number_input("WRRF k", 0, value=60, step=1, key="wrrf_k")
        
        st.markdown("### 🎯 Selected Item")
        if st.session_state.selected_frame_id:
            st.info(f"**{st.session_state.selected_frame_id}**")
            
            # SỬA: Dùng 'get_image_info_cached' thay vì 'system.get_image_info'
            info = get_image_info_cached(system, st.session_state.selected_frame_id.replace('/', '/keyframes/'))
            st.caption(f"Video: {info['folder_name']}")
            st.caption(f"Frame: {info['frame_index']}")
            st.caption(f"Time: {info['pts_time']} ({info['pts_time_seconds']:.2f}s)")
            
            st.markdown("#### 🔍 Neighbor Search")
            num_neighbors = st.slider("Neighbors (each side)", 1, 50, 5, key="num_neighbors_sidebar")
            
            if st.button("Find Neighbors", use_container_width=True):
                neighbor_frames, error_msg = system.search_neighbors(
                    st.session_state.selected_frame_id,
                    num_neighbors
                )
                if error_msg:
                    st.error(error_msg)
                    st.session_state.neighbor_results = []
                else:
                    st.session_state.neighbor_results = neighbor_frames
                st.rerun()
        else:
            st.info("Select an image to view details")

        # ===================================
        # DRES SUBMISSION UI
        # ===================================
        st.markdown("---")
        st.markdown("## 🏆 DRES Submission")
        
        if st.button("Login to DRES (Team 04)", use_container_width=True):
            dres_login(DRES_USERNAME, DRES_PASSWORD)
            
        if st.button("Get Active Evaluation ID", use_container_width=True):
            dres_get_evaluation_id(st.session_state.dres_session_id)
        
        if st.session_state.dres_session_id:
            st.caption(f"Session: ...{st.session_state.dres_session_id[-6:]}")
        if st.session_state.dres_evaluation_id:
            st.caption(f"Evaluation: {st.session_state.dres_evaluation_id}")

        kis_tab, qa_tab, trake_tab = st.tabs(["KIS", "QA", "TRAKE"])

        with kis_tab:
            st.text_input("KIS Input (e.g., K16_V018-12467)", key="kis_input")
            
            if st.button("Submit KIS", use_container_width=True):
                try:
                    text = st.session_state.kis_input.strip()
                    match = re.match(r'(.+?)-(\d+)$', text) # Regex to parse "VIDEO-TIME"
                    
                    if not match:
                        st.error("Invalid KIS format. Use: VIDEO_ID-TIME(ms)")
                    else:
                        video_id = match.group(1).strip()
                        time_ms = int(match.group(2).strip())
                        
                        payload = {
                            "answerSets": [{
                                "answers": [{
                                    "mediaItemName": video_id,
                                    "start": time_ms,
                                    "end": time_ms  # End == Start
                                }]
                            }]
                        }
                        dres_submit(st.session_state.dres_session_id, 
                                    st.session_state.dres_evaluation_id, 
                                    payload)
                except Exception as e:
                    st.error(f"Error formatting KIS: {e}")

        with qa_tab:
            st.text_input("QA Answer", key="qa_answer")
            st.text_input("QA Input (e.g., K16_V018-12467)", key="qa_input")
            
            if st.button("Submit QA", use_container_width=True):
                try:
                    answer = st.session_state.qa_answer.strip()
                    text = st.session_state.qa_input.strip()
                    
                    match = re.match(r'(.+?)-(\d+)$', text) # Regex to parse "VIDEO-TIME"

                    if not match or not answer:
                        st.error("Invalid QA format. Fill answer AND input. Use: VIDEO_ID-TIME(ms)")
                    else:
                        video_id = match.group(1).strip()
                        time_ms = int(match.group(2).strip())
                    
                        qa_text = f"QA-{answer}-{video_id}-{time_ms}"
                        payload = {
                            "answerSets": [{
                                "answers": [{"text": qa_text}]
                            }]
                        }
                        dres_submit(st.session_state.dres_session_id, 
                                    st.session_state.dres_evaluation_id, 
                                    payload)
                except Exception as e:
                    st.error(f"Error formatting QA: {e}")

        with trake_tab:
            st.text_input("TRAKE Input (e.g., K16_V018-12467, 13478, 4687)", key="trake_input")
            
            if st.button("Submit TRAKE", use_container_width=True):
                try:
                    text = st.session_state.trake_input.strip()
                    # Regex to parse "VIDEO-FRAME1, FRAME2, ..."
                    match = re.match(r'(.+?)-([\d,\s]+)$', text) 
                    
                    if not match:
                        st.error("Invalid TRAKE format. Use: VIDEO_ID-FRAME1, FRAME2,...")
                    else:
                        video_id = match.group(1).strip()
                        frames_str = match.group(2)
                        # Clean up the frame string
                        frames = [f.strip() for f in frames_str.split(',') if f.strip()]
                    
                        if not video_id or not frames:
                            st.error("Invalid TRAKE data")
                        else:
                            trake_text = f"TR-{video_id}-{','.join(frames)}"
                            payload = {
                                "answerSets": [{
                                    "answers": [{"text": trake_text}]
                                }]
                            }
                            dres_submit(st.session_state.dres_session_id, 
                                        st.session_state.dres_evaluation_id, 
                                        payload)
                except Exception as e:
                    st.error(f"Error formatting TRAKE: {e}")
        
        # === MỚI: KHU VỰC HIỂN THỊ CÂU HỎI (DASHBOARD) ===
        st.sidebar.markdown("---")
        st.sidebar.markdown("## 📖 Truy Vấn Hiện Tại")

        # Nút bấm để gọi API
        if st.sidebar.button("Tải Câu Hỏi Hiện Tại (Get Current Task)", use_container_width=True):
            if st.session_state.dres_session_id and st.session_state.dres_evaluation_id:
                # Gọi hàm mới
                task_data = dres_get_current_task(
                    st.session_state.dres_session_id,
                    st.session_state.dres_evaluation_id
                )
                # Lưu kết quả vào session_state để giữ lại
                st.session_state.current_task_data = task_data
                st.rerun() # Chạy lại để hiển thị
            else:
                st.sidebar.error("Vui lòng Login và Get Evaluation ID trước.")
        
        # "Dashboard trắng" để hiển thị kết quả
        if 'current_task_data' in st.session_state and st.session_state.current_task_data:
            data = st.session_state.current_task_data
            
            with st.sidebar.container(border=True): # Đây chính là "dashboard trắng"
                task_name = data.get('taskName', 'N/A')
                task_type = data.get('taskType', 'N/A')
                time_left = data.get('remainingTime', 0)
                
                st.markdown(f"**Task:** `{task_name}` (`{task_type}`)")
                st.markdown(f"**Thời gian còn lại:** {int(time_left / 1000)} giây") # Chuyển ms sang s
                st.markdown("---")
                st.markdown("#### Gợi ý (Hints):")
                
                hints = data.get('hints', [])
                if not hints:
                    st.caption("Chưa có gợi ý cho task này (hoặc task đã kết thúc).")
                
                for hint in hints:
                    hint_type = hint.get('type')
                    content = hint.get('content')
                    if hint_type == 'TEXT':
                        # Hiển thị câu hỏi (ví dụ: "Một con robot có 4 chân...")
                        st.info(f"{content}")
                    elif hint_type == 'VIDEO':
                        # Thử hiển thị video nếu là URL
                        if content.startswith('http'):
                            st.video(content)
                        else:
                            st.warning(f"Video Hint (không xem được): {content}")
                    elif hint_type == 'IMAGE':
                        # Thử hiển thị ảnh nếu là URL
                        if content.startswith('http'):
                            st.image(content)
                        else:
                            st.warning(f"Image Hint (không xem được): {content}")
        else:
            st.sidebar.caption("Chưa có thông tin task. Hãy nhấn nút 'Tải'.")
        # === KẾT THÚC KHU VỰC MỚI ===

    
    # Display neighbor results if available
    if st.session_state.neighbor_results:
        st.markdown("---")
        st.subheader(f"🔍 Neighbors for '{st.session_state.selected_frame_id}'")
        display_gallery_with_video(
            st.session_state.neighbor_results,
            system,
            key_prefix="neighbor",
            cols_per_row=st.session_state.cols_per_row,
            highlight_target_id=st.session_state.selected_frame_id,
            show_video_player=False  # Optimization
        )
        st.markdown("---")
    
    # Main tabs
    # TÍNH NĂNG MỚI: Thêm tab6
    tab1, tab2, tab3, tab4, tab5, tab6 = st.tabs([
        "🔍 Hybrid Search",
        "🖼️ Image Search",
        "📝 OCR Search",
        "🎤 Transcript Search",
        "📁 Browser",
        "⛓️ Trake Search"
    ])
    
    # TAB 1: Hybrid Search
    with tab1:
        st.markdown("### Hybrid/Text/Object Search")
        
        all_object_tags = []
        if es_client:
            all_object_tags = get_all_tags(es_client)

        available_modes = []
        if milvus_collection:
            available_modes.append("Vector Search")
        if es_client and all_object_tags:
            available_modes.append("Object Search")
        if milvus_collection and es_client and nlp_model:
            available_modes.insert(0, "Hybrid Search")
        
        if not available_modes:
            st.error("No search mode available")
        else:
            search_mode = st.radio(
                "Search Mode:",
                options=available_modes,
                key="search_mode",
                horizontal=True
            )
            
            if "Object" in search_mode and nlp_model:
                nlp_text = st.text_input(
                    "Extract objects from description:",
                    key="nlp_helper_input",
                    placeholder="e.g., A cat sitting on a red chair"
                )
                
                if st.button("Extract Objects"):
                    if nlp_text:
                        # MỚI: Logic dịch
                        translated_nlp_text = nlp_text
                        if st.session_state.translate_to_en:
                            with st.spinner(f"Đang dịch '{nlp_text}'..."):
                                translated_nlp_text = translate_to_english(nlp_text)
                                if translated_nlp_text != nlp_text:
                                    st.info(f"Đã dịch: '{translated_nlp_text}'")
                        
                        extracted = parse_english_text(translated_nlp_text, nlp_model) # Dùng text đã dịch
                        valid_objects = [obj for obj in extracted if obj in all_object_tags]
                        st.session_state.nlp_extracted_objects = valid_objects
                        if valid_objects:
                            st.success(f"Extracted: {valid_objects}")
                        else:
                            st.warning("No valid objects extracted")
                    else:
                        st.session_state.nlp_extracted_objects = []
            
            with st.form(key="hybrid_search_form"):
                search_query = ""
                selected_objects = []
                
                if "Hybrid" in search_mode or "Vector" in search_mode:
                    search_query = st.text_input(
                        "Text Query:",
                        key="text_query_input",
                        placeholder="Describe the image..."
                    )
                
                if "Object" in search_mode:
                    selected_objects = st.multiselect(
                        "Select objects (must match all):",
                        options=all_object_tags,
                        default=st.session_state.nlp_extracted_objects,
                        key="object_multiselect"
                    )
                
                submitted = st.form_submit_button("🔍 Search", type="primary")
            
            if submitted:
                st.session_state.nlp_extracted_objects = []
                
                top_k = st.session_state.top_k
                kf_filter = st.session_state.kf_filter
                vid_filter = st.session_state.vid_filter
                
                results = []
                run_search = False
                
                if (search_mode in ["Vector Search", "Hybrid Search"]) and search_query:
                    run_search = True
                elif search_mode == "Object Search" and selected_objects:
                    run_search = True
                else:
                    st.warning("Please enter query or select objects")
                
                if run_search:
                    with st.spinner(f"Searching with {search_mode}..."):
                        start_time = time.time()
                        
                        # MỚI: Logic dịch
                        translated_search_query = search_query
                        if (search_mode in ["Vector Search", "Hybrid Search"]) and search_query:
                            if st.session_state.translate_to_en:
                                with st.spinner(f"Đang dịch '{search_query}'..."):
                                    translated_search_query = translate_to_english(search_query)
                                    if translated_search_query != search_query:
                                        st.info(f"Đã dịch: '{translated_search_query}'")
                        
                        if search_mode == "Vector Search":
                            query_vector = encode_text_query(translated_search_query, model, tokenizer, device) # Dùng text đã dịch
                            results = search_milvus_by_vector(
                                milvus_collection, query_vector, top_k,
                                kf_filter, vid_filter, system
                            )
                        
                        elif search_mode == "Object Search":
                            st.info(f"Searching objects: {selected_objects}")
                            results = search_elasticsearch(
                                es_client, selected_objects, top_k,
                                kf_filter, vid_filter, system, match_mode='must'
                            )
                        
                        elif search_mode == "Hybrid Search":
                            k_fusion_vector = top_k * FILTER_FETCH_FACTOR
                            if kf_filter != "All":
                                k_fusion_vector = min(MAX_FILTER_FETCH, k_fusion_vector)
                            else:
                                k_fusion_vector = min(max(top_k, 200), MAX_FILTER_FETCH)
                            
                            k_fusion_object = min(MAX_FILTER_FETCH, top_k * FILTER_FETCH_FACTOR)

                            query_vector = encode_text_query(translated_search_query, model, tokenizer, device) # Dùng text đã dịch
                            
                            vector_results = search_milvus_by_vector(
                                milvus_collection, query_vector, k_fusion_vector,
                                kf_filter, vid_filter, system
                            )
                            
                            objects = parse_english_text(translated_search_query, nlp_model) # Dùng text đã dịch
                            object_results = []
                            if objects:
                                st.info(f"Extracted objects: {objects}")
                                object_results = search_elasticsearch(
                                    es_client, objects, k_fusion_object,
                                    kf_filter, vid_filter, system, match_mode='should'
                                )
                            
                            sources = []
                            weights = []
                            if vector_results:
                                sources.append(vector_results)
                                weights.append(st.session_state.w_vector)
                            if object_results:
                                sources.append(object_results)
                                weights.append(st.session_state.w_object)
                            
                            if sources:
                                fused_results = rank_wrrf(sources, weights, k=st.session_state.wrrf_k)
                                results = fused_results[:top_k]
                            
                        end_time = time.time()
                        st.success(f"✅ Found {len(results)} results in {end_time - start_time:.2f}s")
                        st.session_state.search_results = results
        
        st.markdown("---")
        if st.session_state.search_results:
            st.subheader("Search Results")
            display_gallery_with_video(
                st.session_state.search_results,
                system,
                key_prefix="search",
                cols_per_row=st.session_state.cols_per_row,
                show_video_player=True
            )
    
    # TAB 2: Image Search
    with tab2:
        st.markdown("### Image-to-Image Search")
        
        if not milvus_collection or not model or not image_transform:
            st.error("Image search requires Milvus and Perception Model")
        else:
            with st.form(key="image_search_form"):
                input_method = st.radio(
                    "Input Method:",
                    options=["Upload Image", "Image URL"],
                    horizontal=True,
                    key="img_search_method"
                )
                
                uploaded_file = None
                image_url = ""
                
                if input_method == "Upload Image":
                    uploaded_file = st.file_uploader(
                        "Choose image:",
                        type=["jpg", "jpeg", "png"],
                        key="img_upload"
                    )
                else:
                    image_url = st.text_input("Image URL:", key="img_url")
                
                submitted = st.form_submit_button("🔍 Search", type="primary")
            
            if submitted:
                input_image = None
                
                if input_method == "Upload Image" and uploaded_file:
                    try:
                        input_image = Image.open(io.BytesIO(uploaded_file.read()))
                    except Exception as e:
                        st.error(f"Error loading image: {e}")
                elif input_method == "Image URL" and image_url:
                    try:
                        response = requests.get(image_url, timeout=10)
                        response.raise_for_status()
                        input_image = Image.open(io.BytesIO(response.content))
                    except Exception as e:
                        st.error(f"Error loading URL: {e}")
                
                if input_image:
                    st.image(input_image, caption="Query Image", width=300)
                    
                    with st.spinner("Encoding and searching..."):
                        start_time = time.time()
                        query_vector = encode_image_query(input_image, model, image_transform, device)
                        
                        if query_vector:
                            results = search_milvus_by_vector(
                                milvus_collection, query_vector,
                                st.session_state.top_k,
                                st.session_state.kf_filter,
                                st.session_state.vid_filter,
                                system
                            )
                            end_time = time.time()
                            st.success(f"✅ Found {len(results)} results in {end_time - start_time:.2f}s")
                            st.session_state.image_search_results = results
                        else:
                            st.error("Failed to encode image")
                elif submitted:
                    st.warning("Please provide an image")
        
        st.markdown("---")
        if st.session_state.image_search_results:
            st.subheader("Image Search Results")
            display_gallery_with_video(
                st.session_state.image_search_results,
                system,
                key_prefix="img_search",
                cols_per_row=st.session_state.cols_per_row,
                show_video_player=True
            )
    
    # TAB 3: OCR Search
    with tab3:
        st.markdown("### OCR Text Search")
        
        if not es_client:
            st.error("OCR search requires Elasticsearch connection")
        else:
            all_collections = get_all_collections(es_client)
            
            with st.sidebar:
                st.markdown("#### OCR Filters")
                selected_collections = st.multiselect(
                    "Filter by Collection:",
                    options=all_collections,
                    default=[],
                    key="ocr_collections"
                )
            
            ocr_query = st.text_input(
                "Enter OCR text to search:",
                key="ocr_query",
                placeholder="e.g., HTV7 18:46:27"
            )
            
            if st.button("🔍 Search OCR", type="primary"):
                if not ocr_query:
                    st.warning("Please enter search text")
                else:
                    # MỚI: Logic dịch
                    translated_ocr_query = ocr_query
                    if st.session_state.translate_to_en:
                        with st.spinner(f"Đang dịch '{ocr_query}'..."):
                            translated_ocr_query = translate_to_english(ocr_query)
                            if translated_ocr_query != ocr_query:
                                st.info(f"Đã dịch: '{translated_ocr_query}'")
                                
                    with st.spinner("Searching OCR index..."):
                        try:
                            match_query = {
                                "match": {
                                    "text": {
                                        "query": translated_ocr_query, # Dùng text đã dịch
                                        "fuzziness": "AUTO"
                                    }
                                }
                            }
                            
                            bool_query = {"must": match_query}
                            
                            if selected_collections:
                                bool_query["filter"] = [{
                                    "terms": {"collection_id": selected_collections}
                                }]
                            
                            query_body = {
                                "query": {"bool": bool_query},
                                "size": st.session_state.top_k,
                                "_source": ["image_path", "video_id", "collection_id"]
                            }
                            
                            response = es_client.search(index=OCR_INDEX_NAME, body=query_body)
                            hits = response.get('hits', {}).get('hits', [])
                            
                            if not hits:
                                st.info("No results found")
                            else:
                                ocr_results = []
                                for hit in hits:
                                    source = hit['_source']
                                    image_path = source.get('image_path', '')
                                    
                                    if image_path.startswith('/'):
                                        image_path = image_path[1:]
                                    
                                    ocr_results.append({
                                        'path': image_path,
                                        'score': hit['_score']
                                    })
                                
                                st.session_state.ocr_results = ocr_results
                                st.success(f"✅ Found {len(ocr_results)} results")
                        
                        except Exception as e:
                            st.error(f"OCR search error: {e}")
            
            st.markdown("---")
            if st.session_state.ocr_results:
                st.subheader("OCR Search Results")
                display_gallery_with_video(
                    st.session_state.ocr_results,
                    system,
                    key_prefix="ocr",
                    cols_per_row=st.session_state.cols_per_row,
                    show_video_player=True
                )
    
    # TAB 4: Transcript Search
    with tab4:
        st.markdown("### Video Transcript Search")
        
        if not es_client:
            st.error("Transcript search requires Elasticsearch")
        else:
            all_video_ids = get_all_video_ids(es_client)
            video_metadata = load_video_metadata()
            
            with st.sidebar:
                st.markdown("#### Transcript Filters")
                selected_videos = st.multiselect(
                    "Filter by Video ID:",
                    options=all_video_ids,
                    default=[],
                    key="transcript_videos"
                )
            
            transcript_query = st.text_input(
                "Search in transcripts:",
                key="transcript_query",
                placeholder="e.g., đồng bằng sông cửu long"
            )
            
            if st.button("🔍 Search Transcripts", type="primary"):
                if not transcript_query:
                    st.warning("Please enter search text")
                else:
                    # MỚI: Logic dịch
                    translated_transcript_query = transcript_query
                    if st.session_state.translate_to_en:
                        with st.spinner(f"Đang dịch '{transcript_query}'..."):
                            translated_transcript_query = translate_to_english(transcript_query)
                            if translated_transcript_query != transcript_query:
                                st.info(f"Đã dịch: '{translated_transcript_query}'")

                    with st.spinner("Searching transcripts..."):
                        try:
                            match_query = {
                                "match": {
                                    "text": {
                                        "query": translated_transcript_query, # Dùng text đã dịch
                                        "fuzziness": "AUTO"
                                    }
                                }
                            }
                            
                            bool_query = {"must": match_query}
                            
                            if selected_videos:
                                bool_query["filter"] = [{
                                    "terms": {"video_id": selected_videos}
                                }]
                            
                            query_body = {
                                "query": {"bool": bool_query},
                                "highlight": {
                                    "fields": {"text": {}},
                                    "pre_tags": ["<mark>"],
                                    "post_tags": ["</mark>"]
                                },
                                "size": st.session_state.top_k
                            }
                            
                            response = es_client.search(index=TRANSCRIPT_INDEX_NAME, body=query_body)
                            hits = response.get('hits', {}).get('hits', [])
                            
                            if not hits:
                                st.info("No transcript results found")
                            else:
                                st.success(f"✅ Found {len(hits)} transcript matches")
                                
                                cols = st.columns(2)
                                for i, hit in enumerate(hits):
                                    with cols[i % 2]:
                                        source = hit['_source']
                                        video_id = source.get('video_id')
                                        start_time = source.get('start', 0) or 0
                                        
                                        metadata = video_metadata.get(video_id, {}) if video_metadata else {}
                                        title = metadata.get('title', video_id)
                                        video_url = metadata.get('video_url')
                                        
                                        highlighted = hit.get('highlight', {}).get('text', [source.get('text')])[0]
                                        
                                        time_in_ms = int(start_time * 1000)
                                        fps = metadata.get("fps", 25.0)
                                        frame_idx = int(start_time * fps)
                                        
                                        with st.container(border=True):
                                            st.subheader(f"🎬 {title}")
                                            st.markdown(f"**Match:** {highlighted}", unsafe_allow_html=True)
                                            st.caption(f"Time: {format_time(start_time)} ({int(start_time)}s)")
                                            
                                            # SỬA: Dùng st.code
                                            st.markdown(f"**Idx:**")
                                            st.code(f"{video_id}-{frame_idx}", language=None)
                                            st.markdown(f"**Time (ms):**")
                                            st.code(f"{video_id}-{time_in_ms}", language=None)
                                            
                                            if video_url:
                                                with st.expander("▶️ Watch Video"):
                                                    st.video(video_url, start_time=int(start_time))

                        except Exception as e:
                            st.error(f"Transcript search error: {e}")
    
    # TAB 5: Browser
    with tab5:
        st.markdown("### Video & Image Browser")
        
        col1, col2 = st.columns(2)
        with col1:
            browse_kf = st.selectbox(
                "Keyframes Folder:",
                options=available_keyframes,
                key="browse_kf"
            )
        
        with col2:
            if browse_kf:
                browse_videos = folder_structure.get(browse_kf, [])
                current_browse = st.session_state.get("browse_video")
                if current_browse not in browse_videos and browse_videos:
                    st.session_state.browse_video = browse_videos[0]
                
                browse_video = st.selectbox(
                    "Video Folder:",
                    options=browse_videos,
                    key="browse_video"
                )
            else:
                browse_video = None
        
        # === SỬA: Tách làm 2 nút ===
        btn_col1, btn_col2 = st.columns(2)

        with btn_col1:
            # Nút 1: Chỉ tải Video Player
            if st.button("📁 Load Video Player", type="primary", use_container_width=True):
                if not browse_kf or not browse_video:
                    st.warning("Vui lòng chọn Keyframe Folder và Video Folder")
                else:
                    # Chỉ set video ID
                    st.session_state.current_browse_video = browse_video
                    # Xóa keyframes cũ (nếu có) để tránh nhầm lẫn
                    if 'browse_results' in st.session_state:
                        st.session_state.browse_results = [] 
                    st.rerun() # Chạy lại để hiển thị video player

        with btn_col2:
            # Nút 2: Chỉ tải Keyframes
            if st.button("🖼️ Load Keyframes", use_container_width=True):
                current_video_to_load = st.session_state.get('current_browse_video')
                
                # Phải chọn video trước
                if not current_video_to_load:
                    st.warning("Vui lòng 'Load Video Player' trước.")
                # Đảm bảo video đang chọn khớp với video đã load
                elif current_video_to_load != browse_video:
                     st.warning(f"Video đã tải ({current_video_to_load}) không khớp với video đang chọn ({browse_video}). Vui lòng 'Load Video Player' lại.")
                else:
                    with st.spinner(f"Loading keyframes for {current_video_to_load}..."):
                        # Đây là logic chậm, chỉ chạy khi nhấn nút này
                        prefix = f"{browse_kf}/keyframes/{current_video_to_load}"
                        paths = sorted([
                            p for p in system._get_structure_paths()
                            if p.startswith(prefix)
                        ])
                        st.session_state.browse_results = [{"path": p, "score": 1.0} for p in paths]
                        st.rerun() # Chạy lại để hiển thị keyframes
        # === KẾT THÚC SỬA ===

        st.markdown("---")
        
        if st.session_state.get('current_browse_video'):
            video_id = st.session_state.current_browse_video
            
            st.markdown(f"### 🎬 Video Player: {video_id}")
            youtube_url = system.youtube_url_map.get(video_id)
            if youtube_url:
                st.video(youtube_url)
            else:
                st.warning("YouTube URL not found")
            
            st.markdown("### ⏱️ Time Converter")
            c1, c2 = st.columns(2)
            
            with c1:
                st.markdown("**Time → Frame**")
                with st.form(key="time_to_frame"):
                    time_input = st.text_input("Time (1m25s or 1:25 or 85.5):", key="time_in")
                    if st.form_submit_button("Calculate"):
                        try:
                            total_sec = None
                            if 'm' in time_input:
                                parts = time_input.replace('s', '').split('m')
                                total_sec = int(parts[0]) * 60 + float(parts[1] if parts[1] else 0)
                            elif ':' in time_input:
                                parts = time_input.split(':')
                                total_sec = int(parts[0]) * 60 + float(parts[1])
                            else:
                                total_sec = float(time_input)
                            
                            if total_sec is not None:
                                fps = system.video_fps_map.get(video_id, 30.0)
                                if fps > 0:
                                    st.success(f"Frame: {round(total_sec * fps)}")
                        except:
                            st.error("Invalid format")
            
            with c2:
                st.markdown("**Frame → Time**")
                with st.form(key="frame_to_time"):
                    frame_input = st.text_input("Frame Index:", key="frame_in")
                    if st.form_submit_button("Calculate"):
                        try:
                            frame_idx = int(frame_input)
                            fps = system.video_fps_map.get(video_id, 30.0)
                            if fps > 0:
                                seconds = frame_idx / fps
                                st.success(f"Time: {format_time(seconds)}")
                        except:
                            st.error("Invalid frame index")
            
            st.markdown("---")
            st.markdown("### 🖼️ Keyframes")
            if st.session_state.get('browse_results'):
                display_gallery_with_video(
                    st.session_state.browse_results,
                    system,
                    key_prefix="browse",
                    cols_per_row=st.session_state.cols_per_row,
                    show_video_player=False, # Optimization
                )
            else:
                # MỚI: Thêm thông báo nếu chưa tải keyframes
                st.caption("Nhấn 'Load Keyframes' (ở trên) để xem thư viện ảnh.")

    # TÍNH NĂNG MỚI: TAB 6 - TRAKE SEARCH
    with tab6:
        st.markdown("### ⛓️ Trake (Sequential Event) Search")
        st.info("Tìm kiếm các sự kiện xảy ra theo đúng thứ tự thời gian (E1 -> E2 -> E3 -> E4).")
        
        if not milvus_collection or not model:
            st.error("Trake Search requires Milvus and Perception Model.")
        else:
            with st.form(key="trake_search_form"):
                st.markdown("#### Giai đoạn 1: Nhập các sự kiện (Macro UI)")
                e1_query = st.text_input(
                    "Sự kiện 1 (E1):",
                    placeholder="e.g., Cảnh sát vẫy tay đưa nghi phạm vào"
                )
                e2_query = st.text_input(
                    "Sự kiện 2 (E2) (Tùy chọn):",
                    placeholder="e.g., Cảnh sát chỉ tay vào vị trí"
                )
                e3_query = st.text_input(
                    "Sự kiện 3 (E3) (Tùy chọn):",
                    placeholder="e.g., Cảnh sát khom người xuống"
                )
                
                e4_query = st.text_input(
                    "Sự kiện 4 (E4) (Tùy chọn):",
                    placeholder="Sự kiện cuối cùng (nếu có)"
                )
                
                st.markdown("#### Giai đoạn 2: Cấu hình tìm kiếm (Macro Backend)")
                c1, c2 = st.columns(2)
                with c1:
                    trake_search_k = st.slider(
                        "Số ứng viên cho mỗi sự kiện (K)", 10, 500, 100,
                        help="Tìm K kết quả tốt nhất cho E1, E2, E3 trước khi lọc thứ tự."
                    )
                with c2:
                    max_sequences = st.slider(
                        "Số chuỗi kết quả tối đa", 1, 50, 10,
                        help="Số lượng chuỗi E1->E2->E3 tối đa sẽ hiển thị."
                    )
                
                submitted_trake = st.form_submit_button("🔍 Search Sequences", type="primary")

            if submitted_trake:
                queries = [q for q in [e1_query, e2_query, e3_query, e4_query] if q.strip()]
                
                kf_filter = st.session_state.kf_filter
                vid_filter = st.session_state.vid_filter
                
                if len(queries) < 1:
                    st.warning("Vui lòng nhập ít nhất 1 sự kiện (E1).")
                else:
                    # MỚI: Logic dịch
                    translated_queries = []
                    if st.session_state.translate_to_en:
                        with st.spinner("Đang dịch các sự kiện Trake..."):
                            start_translate = time.time()
                            for q in queries:
                                translated_q = translate_to_english(q)
                                translated_queries.append(translated_q)
                            end_translate = time.time()
                            st.info(f"Đã dịch {len(translated_queries)} câu trong {end_translate - start_translate:.2f}s: {translated_queries}")
                    else:
                        translated_queries = queries # Dùng bản gốc
                
                    with st.spinner(f"Đang tìm kiếm {len(translated_queries)} sự kiện..."):
                        all_results_list = []
                        search_start_time = time.time()
                        
                        for i, query in enumerate(translated_queries): # Dùng text đã dịch
                            st.write(f"Đang tìm E{i+1}: '{query}'...")
                            query_vector = encode_text_query(query, model, tokenizer, device)
                            event_results = search_milvus_for_trake(
                                milvus_collection, query_vector, trake_search_k, system
                            )
                            if not event_results:
                                st.error(f"Không tìm thấy kết quả nào cho E{i+1}. Dừng tìm kiếm.")
                                all_results_list = [] # Xóa kết quả
                                break
                            all_results_list.append(event_results)
                    
                    if all_results_list:
                        st.write("Đã tìm thấy các ứng viên. Đang lọc thứ tự thời gian...")
                        sequences = find_temporal_sequences(
                            all_results_list, 
                            max_sequences, 
                            kf_filter, 
                            vid_filter
                        )
                        end_time = time.time()
                        
                        st.success(f"✅ Đã tìm thấy {len(sequences)} chuỗi hợp lệ trong {end_time - search_start_time:.2f}s.")
                        st.session_state.trake_sequences_results = sequences
            
            st.markdown("---")
            st.markdown("#### Giai đoạn 3: Phân tích kết quả (Manual Micro-Analysis)")
            
            if not st.session_state.trake_sequences_results:
                st.info("Chưa có kết quả chuỗi. Vui lòng chạy tìm kiếm ở trên.")
            else:
                st.write(f"Hiển thị {len(st.session_state.trake_sequences_results)} chuỗi kết quả:")
                
                # This is the main loop for each sequence
                for i, seq in enumerate(st.session_state.trake_sequences_results):
                    st.subheader(f"Chuỗi kết quả #{i+1}")
                    
                    total_score = sum(res['score'] for res in seq)
                    st.caption(f"Video: {seq[0]['info']['folder_name']} | Tổng điểm: {total_score:.4f}")
                    
                    # === MODIFICATION START ===
                    # 1. Tạo các cột cho chuỗi này
                    #    Số cột sẽ bằng số sự kiện trong chuỗi (ví dụ: 1, 2, 3, 4)
                    num_events = len(seq)
                    cols = st.columns(num_events) 
                    
                    # 2. Lặp qua từng sự kiện và gán vào từng cột
                    for j, res in enumerate(seq):
                        with cols[j]:
                            st.markdown(f"**Sự kiện E{j+1}** (Score: {res['score']:.4f})")
                            
                            # 3. Hiển thị thủ công từng ảnh và thông tin
                            #    (Đây là logic rút gọn từ display_gallery_with_video)
                            rel_path = res["path"]
                            full_path = system.get_full_path(rel_path)
                            
                            if full_path and full_path.exists():
                                try:
                                    # Tải ảnh
                                    img = Image.open(str(full_path)).convert("RGB")
                                    buf = io.BytesIO()
                                    img.save(buf, format="JPEG")
                                    byte_im = buf.getvalue()

                                    # Lấy metadata
                                    img_info = res['info'] # Info đã được đính kèm
                                    video_id = img_info['folder_name']
                                    frame_name = img_info['image_name']
                                    frame_id = f"{video_id}/{frame_name}"
                                    
                                    # MỚI: Thêm 2 giá trị copy
                                    format_index_str, format_time_str, base_video_url, start_time_sec, time_ms, frame_idx, copy_idx_str, copy_time_str = system.calculate_metadata(
                                        video_id, f"{frame_name}.jpg"
                                    )

                                    # Hiển thị
                                    st.image(byte_im, use_container_width=True)
                                    
                                    # === SỬA: Hiển thị Idx/Time với st.code ===
                                    st.markdown(f"**Idx:**")
                                    st.code(copy_idx_str, language=None)
                                    
                                    st.markdown(f"**Time (ms):**")
                                    st.code(copy_time_str, language=None)
                                    # === KẾT THÚC SỬA ===
                                    
                                    # Nút Select
                                    if st.button("Select", key=f"trake_{i}_{j}_btn"):
                                        st.session_state.selected_frame_id = frame_id
                                        st.rerun()

                                    # Trình phát video
                                    if base_video_url and start_time_sec is not None:
                                        with st.expander("▶️ Watch Video"):
                                            st.video(base_video_url, start_time=start_time_sec)

                                except Exception as e:
                                    st.error(f"Error displaying trake item: {e}")
                            else:
                                st.warning(f"Image path not found: {rel_path}")

                    # === MODIFICATION END ===
                    st.markdown("---") # Phân cách giữa các chuỗi


if __name__ == "__main__":
    if not CLIP_AVAILABLE:
        st.error("❌ Thiếu thư viện 'open_clip_torch'. Cài bằng: pip install open_clip_torch")
    elif not TRANSLATE_AVAILABLE:
        st.error("❌ Thiếu 'googletrans'. Cài bằng: pip install googletrans==4.0.0-rc1")
    else:
        main_ui()


# Ver 8

In [18]:
%%writefile app.py
# -*- coding: utf-8 -*-
import streamlit as st
import torch
import os
import json
import time
from PIL import Image, ImageDraw
import numpy as np
from pathlib import Path
import pandas as pd
import glob
import math
import requests
import io
import re

# Page config
try:
    st.set_page_config(layout="wide", page_title="🤖 ATLAS Retrieval")
except st.errors.StreamlitAPIException:
    pass

# Imports
try:
    import spacy
    from elasticsearch import Elasticsearch
    from pymilvus import connections, utility, Collection
    from kaggle_secrets import UserSecretsClient
    HYBRID_AVAILABLE = True
except ImportError:
    HYBRID_AVAILABLE = False
    UserSecretsClient = None

# MỚI: Thêm thư viện Dịch
try:
    import googletrans
    from googletrans import Translator
    TRANSLATE_AVAILABLE = True
except ImportError:
    TRANSLATE_AVAILABLE = False
    Translator = None

# Model imports
try:
    import core.vision_encoder.pe as pe
    import core.vision_encoder.transforms as transforms
    PERCEPTION_AVAILABLE = True
except ImportError:
    PERCEPTION_AVAILABLE = False

# ==============================================================================
# CONFIGURATION
# ==============================================================================

CACHE_PATH = "/kaggle/input/image-path/image_paths_cache.json"
COLLECTION_NAME = "my_image_search_v1"
VECTOR_FIELD = "embedding_vector"
ID_FIELD = "image_path_id"
ES_ENDPOINT_URL = "https://my-elasticsearch-project-a0cb68.es.us-east-1.aws.elastic.cloud:443"
ES_INDEX_NAME = "objects"
ES_OBJECT_FIELD = "objects"
ES_IMAGE_ID_FIELD = "image_id"

# OCR Configuration
OCR_INDEX_NAME = "ocr_index"

# Transcript Configuration
TRANSCRIPT_INDEX_NAME = "transcripts_index"
YOUTUBE_MAP_PATH = '/kaggle/input/youtube-map/youtube_map.json'

# Paths
IMAGES_ROOTS = [
    "/kaggle/input/data-aic-2025-tnt",
    "/kaggle/input/aic-dataset-2025-s2"
]
MAP_KEYFRAMES_ROOTS = [
    "/kaggle/input/metadata-aic-2025/map-keyframes-aic25-b1/map-keyframes",
    "/kaggle/input/aic-dataset-2025-s2/map-keyframes-b2/map-keyframes-b2"
]
VIDEOS_ROOTS = ["/kaggle/input/video-aic-2025"]
MEDIA_INFO_ROOTS = [
    "/kaggle/input/metadata-aic-2025/media-info-aic25-b1/media-info",
    "/kaggle/input/aic-dataset-2025-s2/media-info-aic25-b2/media-info"
]

PERCEPTION_MODEL_NAME = "PE-Core-G14-448"
# SỬA LỖI: Giới hạn topk của Milvus là 1024
MAX_FILTER_FETCH = 1024 
FILTER_FETCH_FACTOR = 10

# DRES API Configuration
DRES_BASE_URL = "https://eventretrieval.oj.io.vn"
DRES_USERNAME = "team004"
DRES_PASSWORD = "8PAmMMUts8"

# ==============================================================================
# DRES API FUNCTIONS
# ==============================================================================

def dres_login(username, password):
    """Logs into DRES and returns a session ID."""
    login_url = f"{DRES_BASE_URL}/api/v2/login"
    try:
        response = requests.post(login_url, json={"username": username, "password": password}, timeout=10)
        response.raise_for_status()  # Raise exception for bad status codes
        data = response.json()
        if "sessionId" in data:
            st.session_state.dres_session_id = data["sessionId"]
            st.sidebar.success(f"DRES Login OK. SessionID: ...{data['sessionId'][-6:]}")
        else:
            st.sidebar.error("DRES Login Failed: 'sessionId' not in response.")
    except requests.exceptions.RequestException as e:
        st.sidebar.error(f"DRES Login Error: {e}")

def dres_get_evaluation_id(session_id):
    """Gets the active evaluation ID."""
    if not session_id:
        st.sidebar.warning("Must login to DRES first.")
        return
    
    list_url = f"{DRES_BASE_URL}/api/v2/client/evaluation/list"
    try:
        response = requests.get(list_url, params={"session": session_id}, timeout=10)
        response.raise_for_status()
        evaluations = response.json()
        active_eval = None
        for eval in evaluations:
            if eval.get("status") == "ACTIVE":
                active_eval = eval
                break
        
        if active_eval:
            st.session_state.dres_evaluation_id = active_eval["id"]
            st.sidebar.success(f"Active Evaluation ID: {active_eval['id']}")
        else:
            st.sidebar.error("No 'ACTIVE' evaluation found.")
    except requests.exceptions.RequestException as e:
        st.sidebar.error(f"DRES Eval Error: {e}")

# === HÀM ĐƯỢC SỬA ===
def dres_submit(session_id, evaluation_id, payload):
    """Submits a payload to DRES and shows the result (CORRECT/WRONG)."""
    if not session_id or not evaluation_id:
        st.sidebar.error("Missing Session or Evaluation ID. Cannot submit.")
        return

    submit_url = f"{DRES_BASE_URL}/api/v2/submit/{evaluation_id}"
    try:
        response = requests.post(submit_url, params={"session": session_id}, json=payload, timeout=15)
        response.raise_for_status() # Sẽ báo lỗi nếu HTTP status là 4xx hoặc 5xx

        # Nếu code chạy đến đây, HTTP status là 200 (OK)
        data = response.json()
        
        api_status = data.get("status", "UNKNOWN_API_STATUS")
        # Đây là trường quan trọng:
        submission_result = data.get("submission", "UNKNOWN_RESULT") 

        # Hiển thị kết quả với màu sắc
        if submission_result == "CORRECT":
            st.sidebar.success(f"✅ NỘP BÀI THÀNH CÔNG: ĐÚNG (CORRECT)")
        elif submission_result == "PARTIALLY_CORRECT":
            st.sidebar.info(f"👍 NỘP BÀI THÀNH CÔNG: ĐÚNG MỘT PHẦN (PARTIALLY_CORRECT)")
        elif submission_result == "WRONG":
            st.sidebar.error(f"❌ NỘP BÀI THÀNH CÔNG: SAI (WRONG)")
        elif submission_result == "INDETERMINATE":
            st.sidebar.warning(f"🤔 NỘP BÀI THÀNH CÔNG: CHƯA XÁC ĐỊNH (INDETERMINATE)")
        else:
            # Fallback nếu DRES trả về status lạ
            st.sidebar.warning(f"⚠️ Nộp OK, nhưng kết quả không rõ: {submission_result} (API Status: {api_status})")
            st.sidebar.json(data) # In ra toàn bộ JSON để debug

    except requests.exceptions.RequestException as e:
        # Xử lý lỗi HTTP (ví dụ: 400, 401, 503)
        error_message = f"DRES Submit Error: {e}"
        if e.response is not None:
            try:
                # Cố gắng lấy thông báo lỗi chi tiết từ DRES
                error_data = e.response.json()
                error_detail = error_data.get('description', e.response.text)
                st.sidebar.error(f"Submit Failed ({e.response.status_code}): {error_detail}")
            except:
                # Nếu response không phải JSON
                st.sidebar.error(f"Submit Failed ({e.response.status_code}): {e.response.text}")
        else:
            st.sidebar.error(error_message)
    except Exception as e:
        # Xử lý các lỗi khác (ví dụ: lỗi parsing JSON)
        st.sidebar.error(f"Lỗi không xác định: {e}")
# === KẾT THÚC SỬA ===

# === MỚI: HÀM LẤY TASK HIỆN TẠI ===
def dres_get_current_task(session_id, evaluation_id):
    """
    Lấy thông tin task đang hoạt động (bao gồm câu hỏi/hints).
    """
    if not session_id or not evaluation_id:
        st.sidebar.error("Chưa login hoặc chưa có Evaluation ID.")
        return None
    
    # Đây là endpoint API chuẩn của DRES mà client dùng để lấy task
    task_url = f"{DRES_BASE_URL}/api/v2/client/evaluation/currentTask"
    
    try:
        response = requests.get(task_url, params={"session": session_id, "evaluationId": evaluation_id}, timeout=10)
        response.raise_for_status() # Báo lỗi nếu status code là 4xx/5xx
        
        task_data = response.json()
        
        # task_data sẽ có dạng:
        # { "taskName": "tkis-01", "taskType": "KIS", "remainingTime": 290000,
        #   "hints": [
        #     { "type": "TEXT", "content": "Một con robot có 4 chân...", "startTime": 0 },
        #     { "type": "VIDEO", "content": "path/to/video.mp4", "startTime": 30000 }
        #   ]
        # }
        
        return task_data
        
    except requests.exceptions.RequestException as e:
        # Xử lý lỗi (ví dụ: task chưa bắt đầu, hoặc đã kết thúc)
        try:
            error_detail = e.response.json().get('description', e.response.text) if e.response else str(e)
            st.sidebar.error(f"Lỗi khi tải task: {error_detail}")
        except: # Fallback nếu response không phải là JSON
             st.sidebar.error(f"Lỗi khi tải task: {e}")
        return None
# === KẾT THÚC HÀM MỚI ===

# ==============================================================================
# METADATA SYSTEM
# ==============================================================================

class MetadataSystem:
    def __init__(self, images_roots, map_keyframes_roots, videos_roots, media_info_roots):
        self.images_roots = [Path(p) for p in images_roots]
        self.map_keyframes_roots = [Path(p) for p in map_keyframes_roots]
        self.videos_roots = [Path(p) for p in videos_roots]
        self.media_info_roots = [Path(p) for p in media_info_roots]

        self.rel_to_full_path_map = self._load_or_build_path_map()
        self.master_relative_paths = sorted(list(self.rel_to_full_path_map.keys()))
        self.frame_idx_map, self.video_fps_map, self.pts_time_map = self._load_frame_idx_and_fps_maps(self.map_keyframes_roots)
        self.youtube_url_map = self._load_media_info(self.media_info_roots)
        self.video_to_sorted_frames = self._build_sorted_frame_map()
        self.video_path_map = self._map_video_files()

    def _load_or_build_path_map(self):
        if os.path.exists(CACHE_PATH):
            try:
                with open(CACHE_PATH, 'r') as f:
                    return json.load(f)
            except:
                pass
        
        path_map = {}
        for root_path in self.images_roots:
            if os.path.exists(root_path):
                prefix = root_path.name
                for dirpath, _, filenames in os.walk(root_path):
                    for filename in filenames:
                        if filename.lower().endswith(".jpg"):
                            full_path_str = os.path.join(dirpath, filename)
                            rel_path = str(Path(full_path_str).relative_to(root_path))
                            path_map[rel_path] = full_path_str
                            prefixed_path = f"{prefix}/{rel_path}"
                            path_map[prefixed_path] = full_path_str
        
        try:
            os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
            with open(CACHE_PATH, 'w') as f:
                json.dump(path_map, f)
        except:
            pass
            
        return path_map

    def get_full_path(self, rel_path):
        full_path_str = self.rel_to_full_path_map.get(rel_path)
        return Path(full_path_str) if full_path_str else None

    def _format_time(self, seconds):
        if pd.isna(seconds) or seconds < 0:
            return "N/A"
        minutes = int(seconds // 60)
        secs = int(seconds % 60)
        return f"{minutes}:{secs:02d}"

    def _load_frame_idx_and_fps_maps(self, map_roots):
        idx_mapping, fps_mapping, pts_time_mapping = {}, {}, {}
        all_csv_files = []
        for root in map_roots:
            if root.exists():
                all_csv_files.extend(root.glob('**/*.csv'))
                
        for csv_file in all_csv_files:
            try:
                video_id = csv_file.stem
                df = pd.read_csv(csv_file)
                df.columns = [c.strip().replace('#', '') for c in df.columns]
                if 'n' in df.columns and 'frame_idx' in df.columns and 'fps' in df.columns:
                    df.dropna(subset=['n', 'frame_idx', 'fps'], inplace=True)
                    idx_mapping[video_id] = pd.to_numeric(df.set_index('n')['frame_idx'], errors='coerce').to_dict()
                    if not df.empty:
                        fps_mapping[video_id] = float(df['fps'].iloc[0])
                    if 'pts_time' in df.columns:
                        pts_time_mapping[video_id] = pd.to_numeric(df.set_index('n')['pts_time'], errors='coerce').to_dict()
            except:
                continue
        return idx_mapping, fps_mapping, pts_time_mapping

    def _load_media_info(self, media_info_roots):
        url_map = {}
        all_json_files = []
        for root in media_info_roots:
            if root.exists():
                all_json_files.extend(root.glob('**/*.json'))
        for json_file in all_json_files:
            try:
                video_id = json_file.stem
                with open(json_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                if 'watch_url' in data:
                    url_map[video_id] = data['watch_url']
            except:
                continue
        return url_map

    def _map_video_files(self):
        video_map = {}
        video_files = []
        for root in self.videos_roots:
            if os.path.exists(root):
                video_files.extend(glob.glob(os.path.join(root, "**", "*.mp4"), recursive=True))
        for f in video_files:
            video_map[Path(f).stem] = f
        return video_map

    def _get_structure_paths(self):
        dataset_names = [root.name for root in self.images_roots]
        return [p for p in self.master_relative_paths if Path(p).parts[0] not in dataset_names]

    def _build_sorted_frame_map(self):
        video_map, sorted_video_map = {}, {}
        for p in self._get_structure_paths():
            try:
                parts = Path(p).parts
                if len(parts) >= 3:
                    video_id = parts[2]
                    video_map.setdefault(video_id, []).append(p)
            except:
                continue
        
        for vid, paths in video_map.items():
            frames = []
            for p in paths:
                try:
                    frame_num = int(Path(p).stem)
                    idx = self.frame_idx_map.get(vid, {}).get(frame_num, -1)
                    if idx != -1:
                        frames.append((idx, p))
                except:
                    continue
            sorted_video_map[vid] = sorted(frames, key=lambda x: x[0])
        return sorted_video_map

    def get_folder_structure(self):
        structure = {}
        for path in self._get_structure_paths():
            parts = Path(path).parts
            if len(parts) >= 3:
                kf = parts[0]
                vid = parts[2]
                structure.setdefault(kf, []).append(vid)
        for kf in structure:
            structure[kf] = sorted(list(set(structure[kf])))
        return structure

    def get_image_info(self, rel_path):
        try:
            path_parts = Path(rel_path).parts
            kf_idx, vid_idx = 0, 2
            dataset_names = [root.name for root in self.images_roots]
            if path_parts[0] in dataset_names:
                kf_idx, vid_idx = 1, 3

            if len(path_parts) > vid_idx:
                keyframe_folder = path_parts[kf_idx]
                video_id = path_parts[vid_idx]
                frame_num_str = Path(rel_path).stem
                try:
                    frame_num = int(frame_num_str)
                except ValueError:
                    frame_num = -1

                frame_idx = self.frame_idx_map.get(video_id, {}).get(frame_num, "N/A")
                pts_time_seconds = self.pts_time_map.get(video_id, {}).get(frame_num, None)
                pts_time_formatted = self._format_time(pts_time_seconds)
                return {
                    "keyframe_folder": keyframe_folder,
                    "folder_name": video_id,
                    "frame_index": str(frame_idx),
                    "image_name": frame_num_str,
                    "pts_time": pts_time_formatted,
                    "pts_time_seconds": pts_time_seconds if pts_time_seconds is not None else -1 # Thay đổi: trả về -1 nếu None
                }
        except:
            pass
        return {
            "keyframe_folder": "Unknown",
            "folder_name": "Unknown",
            "frame_index": "N/A",
            "image_name": Path(rel_path).stem,
            "pts_time": "N/A",
            "pts_time_seconds": -1 # Thay đổi: trả về -1 nếu None
        }

    # === HÀM ĐƯỢC SỬA ===
    def calculate_metadata(self, video_id, image_filename):
        """Calculate frame index, timestamp (ms), video URL, and FPS"""
        try:
            match = re.search(r'(\d+)\.jpg$', image_filename, re.IGNORECASE)
            if match:
                keyframe_n = int(match.group(1))
            else:
                raise ValueError("Invalid filename format")
        except:
            # Sửa: Trả về 9 giá trị
            return "Format index: Error", "Format time: Error", None, None, 0, 0, "ERROR", "ERROR", 0.0

        try:
            frame_idx = self.frame_idx_map[video_id][keyframe_n]
        except KeyError:
            # Sửa: Trả về 9 giá trị
            return f"Format index: {video_id}-NotFound", f"Format time: {video_id}-NotFound", None, None, 0, 0, "NOT_FOUND", "NOT_FOUND", 0.0

        fps = self.video_fps_map.get(video_id, 25.0) # Lấy FPS
        time_ms = 0
        time_sec = 0
        
        try:
            if fps > 0:
                time_sec_float = frame_idx / fps
                time_ms = int(time_sec_float * 1000)
                time_sec = math.floor(time_sec_float)
        except:
            # Sửa: Trả về 9 giá trị
            format_index = f"{video_id}-{frame_idx}"
            return f"Format index: {format_index}", "Format time: Error", None, None, 0, frame_idx, format_index, "ERROR", fps

        base_url = self.youtube_url_map.get(video_id)
        format_index = f"{video_id}-{frame_idx}"
        format_time = f"{video_id}-{time_ms}(ms)"
        format_time_copyable = f"{video_id}-{time_ms}" # MỚI

        # Sửa: Trả về 9 giá trị
        return f"Idx: {format_index}", f"Time: {format_time}", base_url, time_sec, time_ms, frame_idx, format_index, format_time_copyable, fps
    # === KẾT THÚC SỬA ===

    def search_neighbors(self, frame_id, num_neighbors):
        try:
            video_id, fnum_str = frame_id.replace(" ", "").split('/')
        except:
            return None, "Invalid Frame ID format"
        
        sorted_frames = self.video_to_sorted_frames.get(video_id)
        if not sorted_frames:
            return None, f"Video ID not found: {video_id}"
        
        t_idx = -1
        for i, (_, p) in enumerate(sorted_frames):
            if Path(p).stem == fnum_str:
                t_idx = i
                break

        if t_idx == -1:
            return None, f"Frame '{fnum_str}' not found"
        
        s = max(0, t_idx - num_neighbors)
        e = min(len(sorted_frames), t_idx + num_neighbors + 1)
        return sorted_frames[s:e], None

# ==============================================================================
# CACHING FUNCTIONS
# ==============================================================================

# MỚI: Hàm Dịch (có cache và xử lý lỗi)
@st.cache_data
def translate_to_english(text_query):
    """
    Dịch văn bản sang Tiếng Anh, sử dụng cache và xử lý lỗi an toàn.
    """
    if not TRANSLATE_AVAILABLE:
        st.sidebar.warning("Thư viện googletrans chưa được cài đặt. Bỏ qua dịch.")
        return text_query
    if not text_query or not text_query.strip():
        return text_query
        
    try:
        # Khởi tạo Translator() bên trong hàm
        # (Đối tượng Translator không thể cache an toàn)
        translator = Translator()
        
        # Kiểm tra ngôn ngữ
        detected_lang = translator.detect(text_query).lang
        if detected_lang.lower().startswith('en'):
            return text_query # Đã là Tiếng Anh
        
        # Dịch
        translated = translator.translate(text_query, dest='en')
        if translated and translated.text:
            return translated.text
        return text_query # Fallback
    except Exception as e:
        st.sidebar.warning(f"Lỗi API Dịch: {e}. Sử dụng văn bản gốc.")
        return text_query


@st.cache_resource
def get_secrets():
    if not HYBRID_AVAILABLE or UserSecretsClient is None:
        return {}
    try:
        user_secrets = UserSecretsClient()
        return {
            "MILVUS_API_KEY": user_secrets.get_secret("MILVUS_API_KEY"),
            "MILVUS_ENDPOINT": user_secrets.get_secret("MILVUS_ENDPOINT"),
            "ES_API_KEY": user_secrets.get_secret("ES_API_KEY")
        }
    except Exception as e:
        print(f"Error loading secrets: {e}")
        return {}

@st.cache_resource
def load_perception_model(model_name):
    if not PERCEPTION_AVAILABLE:
        return None, None, None, None
    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = pe.CLIP.from_config(model_name, pretrained=True).to(device).eval()
        tokenizer = transforms.get_text_tokenizer(model.context_length)
        if hasattr(transforms, 'get_image_transform') and hasattr(model, 'image_size'):
            image_transform = transforms.get_image_transform(model.image_size)
        else:
            image_transform = None
        return model, tokenizer, device, image_transform
    except Exception as e:
        print(f"Error loading model: {e}")
        return None, None, None, None

@st.cache_resource
def connect_to_milvus(endpoint, api_key, collection_name):
    if not endpoint or not api_key:
        return None
    try:
        connections.connect("default", uri=endpoint, token=api_key)
        if not utility.has_collection(collection_name):
            return None
        collection = Collection(collection_name)
        collection.load()
        return collection
    except Exception as e:
        print(f"Milvus connection error: {e}")
        return None

@st.cache_resource
def get_es_client(endpoint, api_key):
    if not endpoint or not api_key:
        return None
    try:
        es = Elasticsearch(endpoint, api_key=api_key)
        if es.ping():
            return es
        return None
    except Exception as e:
        print(f"ES connection error: {e}")
        return None

@st.cache_resource
def get_spacy_model():
    try:
        return spacy.load("en_core_web_lg")
    except:
        return None

@st.cache_resource
def load_metadata_system():
    return MetadataSystem(IMAGES_ROOTS, MAP_KEYFRAMES_ROOTS, VIDEOS_ROOTS, MEDIA_INFO_ROOTS)

# TÍNH NĂNG MỚI: Cache hàm get_image_info để tăng tốc Trake Search
@st.cache_data
def get_image_info_cached(_system, rel_path):
    # Hàm _system.get_image_info không an toàn cho cache vì _system là object
    # Thay vào đó, chúng ta sẽ gọi trực tiếp
    return load_metadata_system().get_image_info(rel_path)

@st.cache_data
def get_all_tags(_es_client):
    if not _es_client:
        return []
    try:
        query_body = {
            "size": 0,
            "aggs": {
                "all_tags": {
                    "terms": {
                        "field": ES_OBJECT_FIELD,
                        "size": 1500
                    }
                }
            }
        }
        response = _es_client.search(index=ES_INDEX_NAME, body=query_body)
        tags = [bucket['key'] for bucket in response['aggregations']['all_tags']['buckets']]
        tags.sort()
        return tags
    except:
        return []

@st.cache_data
def get_cached_folder_structure(_system):
    return _system.get_folder_structure()

@st.cache_data
def get_all_collections(_es_client):
    if not _es_client:
        return []
    try:
        body = {
            "size": 0,
            "aggs": {
                "collections": {
                    "terms": {
                        "field": "collection_id",
                        "size": 1000
                    }
                }
            }
        }
        response = _es_client.search(index=OCR_INDEX_NAME, body=body)
        collections = [bucket['key'] for bucket in response['aggregations']['collections']['buckets'] 
                      if bucket['key'] is not None]
        collections.sort()
        return collections
    except:
        return []

@st.cache_data
def get_all_video_ids(_es_client):
    if not _es_client:
        return []
    try:
        body = {
            "size": 0,
            "aggs": {
                "videos": {
                    "terms": {
                        "field": "video_id",
                        "size": 5000
                    }
                }
            }
        }
        response = _es_client.search(index=TRANSCRIPT_INDEX_NAME, body=body)
        videos = [bucket['key'] for bucket in response['aggregations']['videos']['buckets'] 
                 if bucket['key'] is not None]
        videos.sort()
        return videos
    except:
        return []

@st.cache_data
def load_video_metadata():
    if not os.path.exists(YOUTUBE_MAP_PATH):
        return None
    try:
        with open(YOUTUBE_MAP_PATH, 'r', encoding='utf-8') as f:
            return json.load(f)
    except:
        return None

# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================

def parse_english_text(en_text, nlp_model):
    if not en_text or not nlp_model:
        return []
    doc = nlp_model(en_text)
    objects = [token.text.lower() for token in doc if token.tag_ in ['NN', 'NNS']]
    common_words = ['photo', 'image', 'picture', 'person', 'video', 'frame', 'people']
    return list(set([obj for obj in objects if obj not in common_words]))

def format_time(seconds):
    if seconds is None:
        return "00:00"
    return time.strftime('%M:%S', time.gmtime(float(seconds)))

# === HÀM ĐƯỢC SỬA ===
def display_gallery_with_video(results, system, key_prefix, cols_per_row=4, 
                               highlight_target_id=None, show_video_player=True):
    """Enhanced gallery with options to show/hide video"""
    if not results:
        return

    images_data = []
    
    target_video_id, target_frame_name = None, None
    if highlight_target_id:
        try:
            target_video_id, target_frame_name = highlight_target_id.split('/')
        except:
            pass

    for res in results:
        if isinstance(res, tuple):
            frame_idx_from_tuple, rel_path = res
            score_info = {}
        else:
            rel_path = res["path"]
            score_info = res
            frame_idx_from_tuple = None

        full_path = system.get_full_path(rel_path)
        
        if full_path and full_path.exists():
            try:
                img_info = system.get_image_info(rel_path)
                video_id = img_info['folder_name']
                frame_name = img_info['image_name']
                
                frame_id = f"{video_id}/{frame_name}"

                # Calculate metadata
                # Sửa: Nhận 9 giá trị
                format_index_str, format_time_str, base_video_url, start_time_sec, time_ms, frame_idx, copy_idx_str, copy_time_str, fps = system.calculate_metadata(
                    video_id, f"{frame_name}.jpg"
                )

                # Use frame_idx from tuple if available (for neighbor search)
                current_frame_idx = frame_idx_from_tuple if frame_idx_from_tuple is not None else frame_idx

                # Load and process image
                img = Image.open(str(full_path)).convert("RGB")
                
                # Highlight if target
                is_target = (video_id == target_video_id and frame_name == target_frame_name)
                if is_target:
                    draw = ImageDraw.Draw(img)
                    draw.rectangle([(0,0), (img.width-1, img.height-1)], outline="yellow", width=20)

                # Convert to bytes
                buf = io.BytesIO()
                img.save(buf, format="JPEG")
                byte_im = buf.getvalue()

                images_data.append({
                    'bytes': byte_im,
                    'frame_id': frame_id,
                    'video_id': video_id,
                    'frame_idx': current_frame_idx,
                    'time_ms': time_ms,
                    'format_index': format_index_str, # "Idx: K14_V018-13657"
                    'format_time': format_time_str,   # "Time: K14_V018-546280(ms)"
                    'video_url': base_video_url,
                    'start_time': start_time_sec,
                    'is_target': is_target,
                    'score': score_info.get('score'),
                    'copy_idx_str': copy_idx_str,     # "K14_V018-13657"
                    'copy_time_str': copy_time_str,  # "K14_V018-546280"
                    'fps': fps                       # MỚI: Thêm fps
                })

            except:
                pass

    if not images_data:
        return

    st.write(f"📊 Displaying {len(images_data)} results:")
    cols = st.columns(cols_per_row)
    
    for i, data in enumerate(images_data):
        with cols[i % cols_per_row]:
            # Display image
            if data['is_target']:
                st.markdown("### ⭐ TARGET")
            
            st.image(data['bytes'], use_container_width=True)
            
            # === SỬA: Hiển thị Idx/Time với st.code ===
            st.markdown(f"**Idx:**")
            st.code(data['copy_idx_str'], language=None) # Ngôn ngữ None để không highlight
            
            st.markdown(f"**Time (ms):**")
            st.code(data['copy_time_str'], language=None)
            
            # MỚI: Hiển thị FPS
            st.caption(f"FPS: {data['fps']:.2f}")
            # === KẾT THÚC SỬA ===
            
            if data['score'] is not None:
                st.caption(f"Score: {data['score']:.5f}")
            
            # Video player in expander (conditional)
            if show_video_player and data['video_url'] and data['start_time'] is not None:
                with st.expander("▶️ Watch Video"):
                    st.video(data['video_url'], start_time=data['start_time'])
            
            # Select button for neighbor search
            if st.button("Select", key=f"{key_prefix}_btn_{i}"):
                st.session_state.selected_frame_id = data['frame_id']
                st.rerun()
# === KẾT THÚC SỬA HÀM ===


# ==============================================================================
# SEARCH FUNCTIONS
# ==============================================================================

def encode_text_query(text_query, model, tokenizer, device):
    if not model or not tokenizer:
        return []
    with torch.no_grad():
        text_tokens = tokenizer([text_query]).to(device)
        with torch.autocast(device.type, enabled=device.type != 'cpu'):
            _, text_features, _ = model(None, text_tokens)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        return text_features.cpu().numpy().tolist()

def encode_image_query(image, model, image_transforms, device):
    if not model or not image_transforms:
        return []
    if image.mode != 'RGB':
        image = image.convert('RGB')
    try:
        image_tensor = image_transforms(image).unsqueeze(0).to(device)
        with torch.no_grad():
            with torch.autocast(device.type, enabled=device.type != 'cpu'):
                image_features, _, _ = model(image_tensor, None)
            image_features /= image_features.norm(dim=-1, keepdim=True)
            return image_features.cpu().numpy().tolist()
    except Exception as e:
        st.error(f"Image encoding error: {e}")
        return []

def apply_post_filtering(ranked_list, kf_filter, vid_filter, system, top_k):
    if kf_filter == "All":
        return ranked_list[:top_k]

    filtered_list = []
    for item in ranked_list:
        path = item['path']
        info = system.get_image_info(path)
        
        if info['keyframe_folder'] == kf_filter:
            if vid_filter == "All" or info['folder_name'] == vid_filter:
                filtered_list.append(item)
                if len(filtered_list) >= top_k:
                    break
    return filtered_list

def search_milvus_by_vector(collection, query_vector, top_k, kf_filter, vid_filter, system):
    if not collection or not query_vector:
        return []
    
    fetch_k = top_k
    if kf_filter != "All":
        fetch_k = min(MAX_FILTER_FETCH, top_k * FILTER_FETCH_FACTOR)

    search_params = {"metric_type": "L2", "params": {}}
    try:
        results = collection.search(
            data=query_vector,
            anns_field=VECTOR_FIELD,
            param=search_params,
            limit=fetch_k,
            output_fields=[ID_FIELD]
        )
        
        ranked_list = []
        for hit in results[0]:
            similarity = 1.0 - max(0.0, hit.distance)
            ranked_list.append({"path": hit.id, "score": similarity})
        
        # SỬA: Chuyển 'system' sang 'get_image_info_cached'
        # ranked_list sẽ được lọc sau khi có thông tin
        
        # Lấy thông tin cho tất cả các kết quả
        results_with_info = []
        for item in ranked_list:
            info = get_image_info_cached(system, item['path']) # Dùng hàm cache
            item['info'] = info
            results_with_info.append(item)

        # Lọc thủ công (thay thế apply_post_filtering)
        if kf_filter == "All":
            return results_with_info[:top_k]
        
        filtered_list = []
        for item in results_with_info:
            if item['info']['keyframe_folder'] == kf_filter:
                if vid_filter == "All" or item['info']['folder_name'] == vid_filter:
                    filtered_list.append(item)
                    if len(filtered_list) >= top_k:
                        break
        return filtered_list

    except Exception as e:
        st.error(f"Milvus search error: {e}")
        return []

# TÍNH NĂNG MỚI: Hàm tìm kiếm Milvus được tối ưu hóa cho Trake
def search_milvus_for_trake(collection, query_vector, top_k, system):
    """
    Tìm kiếm Milvus và trả về kết quả kèm thông tin (info) đã được cache.
    Không thực hiện post-filtering.
    """
    if not collection or not query_vector:
        return []

    search_params = {"metric_type": "L2", "params": {}}
    try:
        results = collection.search(
            data=query_vector,
            anns_field=VECTOR_FIELD,
            param=search_params,
            limit=top_k, # Lấy chính xác top_k
            output_fields=[ID_FIELD]
        )
        
        ranked_list = []
        for hit in results[0]:
            similarity = 1.0 - max(0.0, hit.distance)
            item = {"path": hit.id, "score": similarity}
            # Lấy thông tin từ cache ngay lập tức
            item['info'] = get_image_info_cached(system, item['path'])
            ranked_list.append(item)
        
        return ranked_list
    except Exception as e:
        st.error(f"Milvus (Trake) search error: {e}")
        return []


def search_elasticsearch(es_client, objects_list, top_k, kf_filter, vid_filter, system, match_mode='should'):
    if not es_client or not objects_list:
        return []

    fetch_k = top_k
    if kf_filter != "All":
        fetch_k = min(MAX_FILTER_FETCH, top_k * FILTER_FETCH_FACTOR)

    try:
        term_queries = [{"match": {ES_OBJECT_FIELD: tag}} for tag in objects_list]
        if match_mode == 'must':
            query_body = {"size": fetch_k, "query": {"bool": {"must": term_queries}}}
        else:
            query_body = {"size": fetch_k, "query": {"bool": {"should": term_queries}}}
        
        response = es_client.search(index=ES_INDEX_NAME, body=query_body)
        
        ranked_list = []
        for hit in response['hits']['hits']:
            try:
                image_id = hit['_source'][ES_IMAGE_ID_FIELD]
            except KeyError:
                continue
            score = hit['_score'] if match_mode == 'should' else 1.0
            ranked_list.append({"path": image_id, "score": score})
        
        # SỬA: Chuyển 'system' sang 'get_image_info_cached'
        # ranked_list sẽ được lọc sau khi có thông tin
        
        # Lấy thông tin cho tất cả các kết quả
        results_with_info = []
        for item in ranked_list:
            info = get_image_info_cached(system, item['path']) # Dùng hàm cache
            item['info'] = info
            results_with_info.append(item)

        # Lọc thủ công (thay thế apply_post_filtering)
        if kf_filter == "All":
            return results_with_info[:top_k]
        
        filtered_list = []
        for item in results_with_info:
            if item['info']['keyframe_folder'] == kf_filter:
                if vid_filter == "All" or item['info']['folder_name'] == vid_filter:
                    filtered_list.append(item)
                    if len(filtered_list) >= top_k:
                        break
        return filtered_list

    except Exception as e:
        st.error(f"ES search error: {e}")
        return []

def rank_wrrf(results_list, weights, k=60):
    if not results_list:
        return []
    if len(results_list) != len(weights):
        weights = [1.0] * len(results_list)

    fused_scores = {}
    original_scores = {}

    for list_index, (results, weight) in enumerate(zip(results_list, weights)):
        for rank, item in enumerate(results, start=1):
            path = item['path']
            rrf_score = weight * (1.0 / (k + rank))
            fused_scores[path] = fused_scores.get(path, 0.0) + rrf_score
            
            if path not in original_scores:
                original_scores[path] = {}
            source_name = "vector" if list_index == 0 else "object"
            original_scores[path][f'{source_name}_score'] = item.get('score', 0.0)
            
            # TÍNH NĂNG MỚI: Mang theo 'info' nếu có
            if 'info' in item:
                original_scores[path]['info'] = item['info']

    final_results = []
    for path, score in fused_scores.items():
        result_item = {"path": path, "score": score}
        result_item.update(original_scores.get(path, {}))
        final_results.append(result_item)

    final_results.sort(key=lambda x: x["score"], reverse=True)
    return final_results

# TÍNH NĂNG MỚI: Hàm logic chính cho Trake Search (Giai đoạn 2)
def find_temporal_sequences(event_results_list, max_sequences, kf_filter="All", vid_filter="All"): # THÊM kf_filter, vid_filter
    """
    Tìm các chuỗi hợp lệ từ danh sách kết quả của các sự kiện.
    event_results_list: List[List[result_dict]], trong đó result_dict đã chứa 'info'.
    """
    valid_sequences = []
    
    if not event_results_list:
        return []
    
    num_events = len(event_results_list)
    
    # 1. Nhóm kết quả theo video_id cho từng sự kiện
    video_event_maps = []
    all_common_video_ids = set()
    
    for i, event_results in enumerate(event_results_list):
        video_map = {}
        video_ids_for_this_event = set()
        for res in event_results:
            video_id = res['info'].get('folder_name')
            keyframe_folder = res['info'].get('keyframe_folder') # MỚI: Lấy keyframe folder
            
            if video_id == "Unknown" or res['info'].get('pts_time_seconds', -1) < 0:
                continue
            
            # === LỌC LOGIC MỚI ===
            if kf_filter != "All" and keyframe_folder != kf_filter:
                continue # Bỏ qua nếu không đúng keyframe folder
            
            if vid_filter != "All" and video_id != vid_filter:
                continue # Bỏ qua nếu không đúng video
            # === KẾT THÚC LỌC LOGIC MỚI ===

            video_map.setdefault(video_id, []).append(res)
            video_ids_for_this_event.add(video_id)
            
        video_event_maps.append(video_map)
        
        if i == 0:
            all_common_video_ids = video_ids_for_this_event
        else:
            all_common_video_ids.intersection_update(video_ids_for_this_event)

    if not all_common_video_ids:
        return []

    # 2. Thực hiện "join" tuần tự
    for video_id in all_common_video_ids:
        # Bắt đầu với các khung hình của sự kiện đầu tiên
        sequences_in_progress = [ [res] for res in video_event_maps[0].get(video_id, []) ]
        
        # Lặp qua các sự kiện còn lại (E2, E3, ...)
        for i in range(1, num_events):
            new_sequences_in_progress = []
            event_frames = video_event_maps[i].get(video_id, [])
            
            if not event_frames: # Nếu video này không có kết quả cho sự kiện i
                sequences_in_progress = [] # Xóa tất cả chuỗi đang xử lý
                break # Dừng xử lý video này
            
            for seq in sequences_in_progress:
                last_event_frame = seq[-1]
                last_timestamp = last_event_frame['info']['pts_time_seconds']
                
                # Tìm tất cả các khung hình E[i] xảy ra SAU E[i-1]
                for current_event_frame in event_frames:
                    current_timestamp = current_event_frame['info']['pts_time_seconds']
                    
                    if current_timestamp > last_timestamp:
                        # Tìm thấy một cặp hợp lệ, tạo chuỗi mới
                        new_seq = seq + [current_event_frame]
                        new_sequences_in_progress.append(new_seq)
            
            sequences_in_progress = new_sequences_in_progress
            if not sequences_in_progress:
                break # Không tìm thấy cặp E[i-1] < E[i] nào, dừng video này

        # Sau khi lặp qua tất cả sự kiện
        valid_sequences.extend(sequences_in_progress)
        
        if len(valid_sequences) >= max_sequences:
            return valid_sequences[:max_sequences]

    # TODO: Sắp xếp 'valid_sequences' dựa trên tổng điểm (score)
    # Tạm thời chỉ trả về theo thứ tự tìm thấy
    return valid_sequences

# ==============================================================================
# MAIN UI
# ==============================================================================

def initialize_state():
    defaults = {
        'search_results': [],
        'image_search_results': [],
        'selected_frame_id': None,
        'neighbor_results': [],
        'ocr_results': [],
        'transcript_results': [],
        'nlp_extracted_objects': [],
        'dres_session_id': None,
        'dres_evaluation_id': None,
        'kis_input': '',
        'qa_answer': '',
        'qa_input': '',
        'trake_input': '',
        'trake_sequences_results': [], # TÍNH NĂNG MỚI: Lưu kết quả Trake
        'current_task_data': None # MỚI: Lưu trữ task hiện tại
    }
    for key, value in defaults.items():
        if key not in st.session_state:
            st.session_state[key] = value

def main_ui():
    st.title("🤖 ATLAS Retrieval")
    
    initialize_state()
    
    # Load resources
    system = load_metadata_system()
    model, tokenizer, device, image_transform = load_perception_model(PERCEPTION_MODEL_NAME)
    
    secrets = get_secrets()
    milvus_collection = None
    es_client = None
    nlp_model = None
    all_object_tags = []
    
    if secrets:
        milvus_collection = connect_to_milvus(
            secrets.get("MILVUS_ENDPOINT"),
            secrets.get("MILVUS_API_KEY"),
            COLLECTION_NAME
        )
        es_client = get_es_client(ES_ENDPOINT_URL, secrets.get("ES_API_KEY"))
        nlp_model = get_spacy_model()
    
    folder_structure = get_cached_folder_structure(system)
    available_keyframes = sorted(list(folder_structure.keys()))
    filter_keyframes = ["All"] + available_keyframes
    
    # Sidebar
    with st.sidebar:
        st.markdown("## ⚙️ Settings")
        
        if st.button("🆕 New Search / Clear All", use_container_width=True):
            # Keep DRES credentials
            session_id = st.session_state.dres_session_id
            eval_id = st.session_state.dres_evaluation_id
            
            # Clear all
            for key in st.session_state.keys():
                del st.session_state[key]
            
            # Re-initialize and restore DRES
            initialize_state()
            st.session_state.dres_session_id = session_id
            st.session_state.dres_evaluation_id = eval_id
            
            st.rerun()
        
        st.markdown("### 📂 Filters")
        st.selectbox("Keyframes Folder", options=filter_keyframes, key="kf_filter")
        
        if st.session_state.kf_filter == "All":
            video_choices = ["All"]
        else:
            video_choices = ["All"] + folder_structure.get(st.session_state.kf_filter, [])
        
        current_vid = st.session_state.get("vid_filter", "All")
        if current_vid not in video_choices:
            st.session_state.vid_filter = "All"
        
        st.selectbox("Video Folder", options=video_choices, key="vid_filter")
        
        # MỚI: Thêm nút dịch
        st.checkbox(
            "Dịch truy vấn sang Tiếng Anh", 
            value=True, # Mặc định là có
            key="translate_to_en",
            help="Tự động dịch văn bản (Tiếng Việt,...) sang Tiếng Anh trước khi tìm kiếm."
        )

        st.slider("Top K Results", 1, 200, 12, 1, key="top_k")
        st.slider("Số cột hiển thị", 1, 10, 4, 1, key="cols_per_row")
        with st.expander("WRRF Settings"):
            st.number_input("Vector Weight", 0.0, value=1.0, step=0.1, key="w_vector")
            st.number_input("Object Weight", 0.0, value=1.0, step=0.1, key="w_object")
            st.number_input("WRRF k", 0, value=60, step=1, key="wrrf_k")
        
        st.markdown("### 🎯 Selected Item")
        if st.session_state.selected_frame_id:
            st.info(f"**{st.session_state.selected_frame_id}**")
            
            # SỬA: Dùng 'get_image_info_cached' thay vì 'system.get_image_info'
            info = get_image_info_cached(system, st.session_state.selected_frame_id.replace('/', '/keyframes/'))
            st.caption(f"Video: {info['folder_name']}")
            st.caption(f"Frame: {info['frame_index']}")
            st.caption(f"Time: {info['pts_time']} ({info['pts_time_seconds']:.2f}s)")
            
            st.markdown("#### 🔍 Neighbor Search")
            num_neighbors = st.slider("Neighbors (each side)", 1, 50, 5, key="num_neighbors_sidebar")
            
            if st.button("Find Neighbors", use_container_width=True):
                neighbor_frames, error_msg = system.search_neighbors(
                    st.session_state.selected_frame_id,
                    num_neighbors
                )
                if error_msg:
                    st.error(error_msg)
                    st.session_state.neighbor_results = []
                else:
                    st.session_state.neighbor_results = neighbor_frames
                st.rerun()
        else:
            st.info("Select an image to view details")

        # ===================================
        # DRES SUBMISSION UI
        # ===================================
        st.markdown("---")
        st.markdown("## 🏆 DRES Submission")
        
        if st.button("Login to DRES (Team 04)", use_container_width=True):
            dres_login(DRES_USERNAME, DRES_PASSWORD)
            
        if st.button("Get Active Evaluation ID", use_container_width=True):
            dres_get_evaluation_id(st.session_state.dres_session_id)
        
        if st.session_state.dres_session_id:
            st.caption(f"Session: ...{st.session_state.dres_session_id[-6:]}")
        if st.session_state.dres_evaluation_id:
            st.caption(f"Evaluation: {st.session_state.dres_evaluation_id}")

        kis_tab, qa_tab, trake_tab = st.tabs(["KIS", "QA", "TRAKE"])

        with kis_tab:
            st.text_input("KIS Input (e.g., K16_V018-12467)", key="kis_input")
            
            if st.button("Submit KIS", use_container_width=True):
                try:
                    text = st.session_state.kis_input.strip()
                    match = re.match(r'(.+?)-(\d+)$', text) # Regex to parse "VIDEO-TIME"
                    
                    if not match:
                        st.error("Invalid KIS format. Use: VIDEO_ID-TIME(ms)")
                    else:
                        video_id = match.group(1).strip()
                        time_ms = int(match.group(2).strip())
                        
                        payload = {
                            "answerSets": [{
                                "answers": [{
                                    "mediaItemName": video_id,
                                    "start": time_ms,
                                    "end": time_ms  # End == Start
                                }]
                            }]
                        }
                        dres_submit(st.session_state.dres_session_id, 
                                    st.session_state.dres_evaluation_id, 
                                    payload)
                except Exception as e:
                    st.error(f"Error formatting KIS: {e}")

        with qa_tab:
            st.text_input("QA Answer", key="qa_answer")
            st.text_input("QA Input (e.g., K16_V018-12467)", key="qa_input")
            
            if st.button("Submit QA", use_container_width=True):
                try:
                    answer = st.session_state.qa_answer.strip()
                    text = st.session_state.qa_input.strip()
                    
                    match = re.match(r'(.+?)-(\d+)$', text) # Regex to parse "VIDEO-TIME"

                    if not match or not answer:
                        st.error("Invalid QA format. Fill answer AND input. Use: VIDEO_ID-TIME(ms)")
                    else:
                        video_id = match.group(1).strip()
                        time_ms = int(match.group(2).strip())
                    
                        qa_text = f"QA-{answer}-{video_id}-{time_ms}"
                        payload = {
                            "answerSets": [{
                                "answers": [{"text": qa_text}]
                            }]
                        }
                        dres_submit(st.session_state.dres_session_id, 
                                    st.session_state.dres_evaluation_id, 
                                    payload)
                except Exception as e:
                    st.error(f"Error formatting QA: {e}")

        with trake_tab:
            st.text_input("TRAKE Input (e.g., K16_V018-12467, 13478, 4687)", key="trake_input")
            
            if st.button("Submit TRAKE", use_container_width=True):
                try:
                    text = st.session_state.trake_input.strip()
                    # Regex to parse "VIDEO-FRAME1, FRAME2, ..."
                    match = re.match(r'(.+?)-([\d,\s]+)$', text) 
                    
                    if not match:
                        st.error("Invalid TRAKE format. Use: VIDEO_ID-FRAME1, FRAME2,...")
                    else:
                        video_id = match.group(1).strip()
                        frames_str = match.group(2)
                        # Clean up the frame string
                        frames = [f.strip() for f in frames_str.split(',') if f.strip()]
                    
                        if not video_id or not frames:
                            st.error("Invalid TRAKE data")
                        else:
                            trake_text = f"TR-{video_id}-{','.join(frames)}"
                            payload = {
                                "answerSets": [{
                                    "answers": [{"text": trake_text}]
                                }]
                            }
                            dres_submit(st.session_state.dres_session_id, 
                                        st.session_state.dres_evaluation_id, 
                                        payload)
                except Exception as e:
                    st.error(f"Error formatting TRAKE: {e}")
        
        # === MỚI: KHU VỰC HIỂN THỊ CÂU HỎI (DASHBOARD) ===
        st.sidebar.markdown("---")
        st.sidebar.markdown("## 📖 Truy Vấn Hiện Tại")

        # Nút bấm để gọi API
        if st.sidebar.button("Tải Câu Hỏi Hiện Tại (Get Current Task)", use_container_width=True):
            if st.session_state.dres_session_id and st.session_state.dres_evaluation_id:
                # Gọi hàm mới
                task_data = dres_get_current_task(
                    st.session_state.dres_session_id,
                    st.session_state.dres_evaluation_id
                )
                # Lưu kết quả vào session_state để giữ lại
                st.session_state.current_task_data = task_data
                st.rerun() # Chạy lại để hiển thị
            else:
                st.sidebar.error("Vui lòng Login và Get Evaluation ID trước.")
        
        # "Dashboard trắng" để hiển thị kết quả
        if 'current_task_data' in st.session_state and st.session_state.current_task_data:
            data = st.session_state.current_task_data
            
            with st.sidebar.container(border=True): # Đây chính là "dashboard trắng"
                task_name = data.get('taskName', 'N/A')
                task_type = data.get('taskType', 'N/A')
                time_left = data.get('remainingTime', 0)
                
                st.markdown(f"**Task:** `{task_name}` (`{task_type}`)")
                st.markdown(f"**Thời gian còn lại:** {int(time_left / 1000)} giây") # Chuyển ms sang s
                st.markdown("---")
                st.markdown("#### Gợi ý (Hints):")
                
                hints = data.get('hints', [])
                if not hints:
                    st.caption("Chưa có gợi ý cho task này (hoặc task đã kết thúc).")
                
                for hint in hints:
                    hint_type = hint.get('type')
                    content = hint.get('content')
                    if hint_type == 'TEXT':
                        # Hiển thị câu hỏi (ví dụ: "Một con robot có 4 chân...")
                        st.info(f"{content}")
                    elif hint_type == 'VIDEO':
                        # Thử hiển thị video nếu là URL
                        if content.startswith('http'):
                            st.video(content)
                        else:
                            st.warning(f"Video Hint (không xem được): {content}")
                    elif hint_type == 'IMAGE':
                        # Thử hiển thị ảnh nếu là URL
                        if content.startswith('http'):
                            st.image(content)
                        else:
                            st.warning(f"Image Hint (không xem được): {content}")
        else:
            st.sidebar.caption("Chưa có thông tin task. Hãy nhấn nút 'Tải'.")
        # === KẾT THÚC KHU VỰC MỚI ===

    
    # Display neighbor results if available
    if st.session_state.neighbor_results:
        st.markdown("---")
        st.subheader(f"🔍 Neighbors for '{st.session_state.selected_frame_id}'")
        display_gallery_with_video(
            st.session_state.neighbor_results,
            system,
            key_prefix="neighbor",
            cols_per_row=st.session_state.cols_per_row,
            highlight_target_id=st.session_state.selected_frame_id,
            show_video_player=False  # Optimization
        )
        st.markdown("---")
    
    # Main tabs
    # TÍNH NĂNG MỚI: Thêm tab6
    tab1, tab2, tab3, tab4, tab5, tab6 = st.tabs([
        "🔍 Hybrid Search",
        "🖼️ Image Search",
        "📝 OCR Search",
        "🎤 Transcript Search",
        "📁 Browser",
        "⛓️ Trake Search"
    ])
    
    # TAB 1: Hybrid Search
    with tab1:
        st.markdown("### Hybrid/Text/Object Search")
        
        all_object_tags = []
        if es_client:
            all_object_tags = get_all_tags(es_client)

        available_modes = []
        if milvus_collection:
            available_modes.append("Vector Search")
        if es_client and all_object_tags:
            available_modes.append("Object Search")
        if milvus_collection and es_client and nlp_model:
            available_modes.insert(0, "Hybrid Search")
        
        if not available_modes:
            st.error("No search mode available")
        else:
            search_mode = st.radio(
                "Search Mode:",
                options=available_modes,
                key="search_mode",
                horizontal=True
            )
            
            if "Object" in search_mode and nlp_model:
                nlp_text = st.text_input(
                    "Extract objects from description:",
                    key="nlp_helper_input",
                    placeholder="e.g., A cat sitting on a red chair"
                )
                
                if st.button("Extract Objects"):
                    if nlp_text:
                        # MỚI: Logic dịch
                        translated_nlp_text = nlp_text
                        if st.session_state.translate_to_en:
                            with st.spinner(f"Đang dịch '{nlp_text}'..."):
                                translated_nlp_text = translate_to_english(nlp_text)
                                if translated_nlp_text != nlp_text:
                                    st.info(f"Đã dịch: '{translated_nlp_text}'")
                        
                        extracted = parse_english_text(translated_nlp_text, nlp_model) # Dùng text đã dịch
                        valid_objects = [obj for obj in extracted if obj in all_object_tags]
                        st.session_state.nlp_extracted_objects = valid_objects
                        if valid_objects:
                            st.success(f"Extracted: {valid_objects}")
                        else:
                            st.warning("No valid objects extracted")
                    else:
                        st.session_state.nlp_extracted_objects = []
            
            with st.form(key="hybrid_search_form"):
                search_query = ""
                selected_objects = []
                
                if "Hybrid" in search_mode or "Vector" in search_mode:
                    search_query = st.text_input(
                        "Text Query:",
                        key="text_query_input",
                        placeholder="Describe the image..."
                    )
                
                if "Object" in search_mode:
                    selected_objects = st.multiselect(
                        "Select objects (must match all):",
                        options=all_object_tags,
                        default=st.session_state.nlp_extracted_objects,
                        key="object_multiselect"
                    )
                
                submitted = st.form_submit_button("🔍 Search", type="primary")
            
            if submitted:
                st.session_state.nlp_extracted_objects = []
                
                top_k = st.session_state.top_k
                kf_filter = st.session_state.kf_filter
                vid_filter = st.session_state.vid_filter
                
                results = []
                run_search = False
                
                if (search_mode in ["Vector Search", "Hybrid Search"]) and search_query:
                    run_search = True
                elif search_mode == "Object Search" and selected_objects:
                    run_search = True
                else:
                    st.warning("Please enter query or select objects")
                
                if run_search:
                    with st.spinner(f"Searching with {search_mode}..."):
                        start_time = time.time()
                        
                        # MỚI: Logic dịch
                        translated_search_query = search_query
                        if (search_mode in ["Vector Search", "Hybrid Search"]) and search_query:
                            if st.session_state.translate_to_en:
                                with st.spinner(f"Đang dịch '{search_query}'..."):
                                    translated_search_query = translate_to_english(search_query)
                                    if translated_search_query != search_query:
                                        st.info(f"Đã dịch: '{translated_search_query}'")
                        
                        if search_mode == "Vector Search":
                            query_vector = encode_text_query(translated_search_query, model, tokenizer, device) # Dùng text đã dịch
                            results = search_milvus_by_vector(
                                milvus_collection, query_vector, top_k,
                                kf_filter, vid_filter, system
                            )
                        
                        elif search_mode == "Object Search":
                            st.info(f"Searching objects: {selected_objects}")
                            results = search_elasticsearch(
                                es_client, selected_objects, top_k,
                                kf_filter, vid_filter, system, match_mode='must'
                            )
                        
                        elif search_mode == "Hybrid Search":
                            k_fusion_vector = top_k * FILTER_FETCH_FACTOR
                            if kf_filter != "All":
                                k_fusion_vector = min(MAX_FILTER_FETCH, k_fusion_vector)
                            else:
                                k_fusion_vector = min(max(top_k, 200), MAX_FILTER_FETCH)
                            
                            k_fusion_object = min(MAX_FILTER_FETCH, top_k * FILTER_FETCH_FACTOR)

                            query_vector = encode_text_query(translated_search_query, model, tokenizer, device) # Dùng text đã dịch
                            
                            vector_results = search_milvus_by_vector(
                                milvus_collection, query_vector, k_fusion_vector,
                                kf_filter, vid_filter, system
                            )
                            
                            objects = parse_english_text(translated_search_query, nlp_model) # Dùng text đã dịch
                            object_results = []
                            if objects:
                                st.info(f"Extracted objects: {objects}")
                                object_results = search_elasticsearch(
                                    es_client, objects, k_fusion_object,
                                    kf_filter, vid_filter, system, match_mode='should'
                                )
                            
                            sources = []
                            weights = []
                            if vector_results:
                                sources.append(vector_results)
                                weights.append(st.session_state.w_vector)
                            if object_results:
                                sources.append(object_results)
                                weights.append(st.session_state.w_object)
                            
                            if sources:
                                fused_results = rank_wrrf(sources, weights, k=st.session_state.wrrf_k)
                                results = fused_results[:top_k]
                            
                        end_time = time.time()
                        st.success(f"✅ Found {len(results)} results in {end_time - start_time:.2f}s")
                        st.session_state.search_results = results
        
        st.markdown("---")
        if st.session_state.search_results:
            st.subheader("Search Results")
            display_gallery_with_video(
                st.session_state.search_results,
                system,
                key_prefix="search",
                cols_per_row=st.session_state.cols_per_row,
                show_video_player=True
            )
    
    # TAB 2: Image Search
    with tab2:
        st.markdown("### Image-to-Image Search")
        
        if not milvus_collection or not model or not image_transform:
            st.error("Image search requires Milvus and Perception Model")
        else:
            with st.form(key="image_search_form"):
                input_method = st.radio(
                    "Input Method:",
                    options=["Upload Image", "Image URL"],
                    horizontal=True,
                    key="img_search_method"
                )
                
                uploaded_file = None
                image_url = ""
                
                if input_method == "Upload Image":
                    uploaded_file = st.file_uploader(
                        "Choose image:",
                        type=["jpg", "jpeg", "png"],
                        key="img_upload"
                    )
                else:
                    image_url = st.text_input("Image URL:", key="img_url")
                
                submitted = st.form_submit_button("🔍 Search", type="primary")
            
            if submitted:
                input_image = None
                
                if input_method == "Upload Image" and uploaded_file:
                    try:
                        input_image = Image.open(io.BytesIO(uploaded_file.read()))
                    except Exception as e:
                        st.error(f"Error loading image: {e}")
                elif input_method == "Image URL" and image_url:
                    try:
                        response = requests.get(image_url, timeout=10)
                        response.raise_for_status()
                        input_image = Image.open(io.BytesIO(response.content))
                    except Exception as e:
                        st.error(f"Error loading URL: {e}")
                
                if input_image:
                    st.image(input_image, caption="Query Image", width=300)
                    
                    with st.spinner("Encoding and searching..."):
                        start_time = time.time()
                        query_vector = encode_image_query(input_image, model, image_transform, device)
                        
                        if query_vector:
                            results = search_milvus_by_vector(
                                milvus_collection, query_vector,
                                st.session_state.top_k,
                                st.session_state.kf_filter,
                                st.session_state.vid_filter,
                                system
                            )
                            end_time = time.time()
                            st.success(f"✅ Found {len(results)} results in {end_time - start_time:.2f}s")
                            st.session_state.image_search_results = results
                        else:
                            st.error("Failed to encode image")
                elif submitted:
                    st.warning("Please provide an image")
        
        st.markdown("---")
        if st.session_state.image_search_results:
            st.subheader("Image Search Results")
            display_gallery_with_video(
                st.session_state.image_search_results,
                system,
                key_prefix="img_search",
                cols_per_row=st.session_state.cols_per_row,
                show_video_player=True
            )
    
    # TAB 3: OCR Search
    with tab3:
        st.markdown("### OCR Text Search")
        
        if not es_client:
            st.error("OCR search requires Elasticsearch connection")
        else:
            all_collections = get_all_collections(es_client)
            
            with st.sidebar:
                st.markdown("#### OCR Filters")
                selected_collections = st.multiselect(
                    "Filter by Collection:",
                    options=all_collections,
                    default=[],
                    key="ocr_collections"
                )
            
            ocr_query = st.text_input(
                "Enter OCR text to search:",
                key="ocr_query",
                placeholder="e.g., HTV7 18:46:27"
            )
            
            if st.button("🔍 Search OCR", type="primary"):
                if not ocr_query:
                    st.warning("Please enter search text")
                else:
                    # MỚI: Logic dịch
                    translated_ocr_query = ocr_query
                    if st.session_state.translate_to_en:
                        with st.spinner(f"Đang dịch '{ocr_query}'..."):
                            translated_ocr_query = translate_to_english(ocr_query)
                            if translated_ocr_query != ocr_query:
                                st.info(f"Đã dịch: '{translated_ocr_query}'")
                                
                    with st.spinner("Searching OCR index..."):
                        try:
                            match_query = {
                                "match": {
                                    "text": {
                                        "query": translated_ocr_query, # Dùng text đã dịch
                                        "fuzziness": "AUTO"
                                    }
                                }
                            }
                            
                            bool_query = {"must": match_query}
                            
                            if selected_collections:
                                bool_query["filter"] = [{
                                    "terms": {"collection_id": selected_collections}
                                }]
                            
                            query_body = {
                                "query": {"bool": bool_query},
                                "size": st.session_state.top_k,
                                "_source": ["image_path", "video_id", "collection_id"]
                            }
                            
                            response = es_client.search(index=OCR_INDEX_NAME, body=query_body)
                            hits = response.get('hits', {}).get('hits', [])
                            
                            if not hits:
                                st.info("No results found")
                            else:
                                ocr_results = []
                                for hit in hits:
                                    source = hit['_source']
                                    image_path = source.get('image_path', '')
                                    
                                    if image_path.startswith('/'):
                                        image_path = image_path[1:]
                                    
                                    ocr_results.append({
                                        'path': image_path,
                                        'score': hit['_score']
                                    })
                                
                                st.session_state.ocr_results = ocr_results
                                st.success(f"✅ Found {len(ocr_results)} results")
                        
                        except Exception as e:
                            st.error(f"OCR search error: {e}")
            
            st.markdown("---")
            if st.session_state.ocr_results:
                st.subheader("OCR Search Results")
                display_gallery_with_video(
                    st.session_state.ocr_results,
                    system,
                    key_prefix="ocr",
                    cols_per_row=st.session_state.cols_per_row,
                    show_video_player=True
                )
    
    # TAB 4: Transcript Search
    with tab4:
        st.markdown("### Video Transcript Search")
        
        if not es_client:
            st.error("Transcript search requires Elasticsearch")
        else:
            all_video_ids = get_all_video_ids(es_client)
            video_metadata = load_video_metadata()
            
            with st.sidebar:
                st.markdown("#### Transcript Filters")
                selected_videos = st.multiselect(
                    "Filter by Video ID:",
                    options=all_video_ids,
                    default=[],
                    key="transcript_videos"
                )
            
            transcript_query = st.text_input(
                "Search in transcripts:",
                key="transcript_query",
                placeholder="e.g., đồng bằng sông cửu long"
            )
            
            if st.button("🔍 Search Transcripts", type="primary"):
                if not transcript_query:
                    st.warning("Please enter search text")
                else:
                    # MỚI: Logic dịch
                    translated_transcript_query = transcript_query
                    if st.session_state.translate_to_en:
                        with st.spinner(f"Đang dịch '{transcript_query}'..."):
                            translated_transcript_query = translate_to_english(transcript_query)
                            if translated_transcript_query != transcript_query:
                                st.info(f"Đã dịch: '{translated_transcript_query}'")

                    with st.spinner("Searching transcripts..."):
                        try:
                            match_query = {
                                "match": {
                                    "text": {
                                        "query": translated_transcript_query, # Dùng text đã dịch
                                        "fuzziness": "AUTO"
                                    }
                                }
                            }
                            
                            bool_query = {"must": match_query}
                            
                            if selected_videos:
                                bool_query["filter"] = [{
                                    "terms": {"video_id": selected_videos}
                                }]
                            
                            query_body = {
                                "query": {"bool": bool_query},
                                "highlight": {
                                    "fields": {"text": {}},
                                    "pre_tags": ["<mark>"],
                                    "post_tags": ["</mark>"]
                                },
                                "size": st.session_state.top_k
                            }
                            
                            response = es_client.search(index=TRANSCRIPT_INDEX_NAME, body=query_body)
                            hits = response.get('hits', {}).get('hits', [])
                            
                            if not hits:
                                st.info("No transcript results found")
                            else:
                                st.success(f"✅ Found {len(hits)} transcript matches")
                                
                                cols = st.columns(2)
                                for i, hit in enumerate(hits):
                                    with cols[i % 2]:
                                        source = hit['_source']
                                        video_id = source.get('video_id')
                                        start_time = source.get('start', 0) or 0
                                        
                                        metadata = video_metadata.get(video_id, {}) if video_metadata else {}
                                        title = metadata.get('title', video_id)
                                        video_url = metadata.get('video_url')
                                        
                                        highlighted = hit.get('highlight', {}).get('text', [source.get('text')])[0]
                                        
                                        time_in_ms = int(start_time * 1000)
                                        fps = metadata.get("fps", 25.0)
                                        frame_idx = int(start_time * fps)
                                        
                                        with st.container(border=True):
                                            st.subheader(f"🎬 {title}")
                                            st.markdown(f"**Match:** {highlighted}", unsafe_allow_html=True)
                                            st.caption(f"Time: {format_time(start_time)} ({int(start_time)}s)")
                                            
                                            # SỬA: Dùng st.code
                                            st.markdown(f"**Idx:**")
                                            st.code(f"{video_id}-{frame_idx}", language=None)
                                            st.markdown(f"**Time (ms):**")
                                            st.code(f"{video_id}-{time_in_ms}", language=None)
                                            # MỚI: Thêm FPS
                                            st.caption(f"FPS: {fps:.2f}")

                                            if video_url:
                                                with st.expander("▶️ Watch Video"):
                                                    st.video(video_url, start_time=int(start_time))

                        except Exception as e:
                            st.error(f"Transcript search error: {e}")
    
    # TAB 5: Browser
    with tab5:
        st.markdown("### Video & Image Browser")
        
        col1, col2 = st.columns(2)
        with col1:
            browse_kf = st.selectbox(
                "Keyframes Folder:",
                options=available_keyframes,
                key="browse_kf"
            )
        
        with col2:
            if browse_kf:
                browse_videos = folder_structure.get(browse_kf, [])
                current_browse = st.session_state.get("browse_video")
                if current_browse not in browse_videos and browse_videos:
                    st.session_state.browse_video = browse_videos[0]
                
                browse_video = st.selectbox(
                    "Video Folder:",
                    options=browse_videos,
                    key="browse_video"
                )
            else:
                browse_video = None
        
        # === SỬA: Tách làm 2 nút ===
        btn_col1, btn_col2 = st.columns(2)

        with btn_col1:
            # Nút 1: Chỉ tải Video Player
            if st.button("📁 Load Video Player", type="primary", use_container_width=True):
                if not browse_kf or not browse_video:
                    st.warning("Vui lòng chọn Keyframe Folder và Video Folder")
                else:
                    # Chỉ set video ID
                    st.session_state.current_browse_video = browse_video
                    # Xóa keyframes cũ (nếu có) để tránh nhầm lẫn
                    if 'browse_results' in st.session_state:
                        st.session_state.browse_results = [] 
                    st.rerun() # Chạy lại để hiển thị video player

        with btn_col2:
            # Nút 2: Chỉ tải Keyframes
            if st.button("🖼️ Load Keyframes", use_container_width=True):
                current_video_to_load = st.session_state.get('current_browse_video')
                
                # Phải chọn video trước
                if not current_video_to_load:
                    st.warning("Vui lòng 'Load Video Player' trước.")
                # Đảm bảo video đang chọn khớp với video đã load
                elif current_video_to_load != browse_video:
                     st.warning(f"Video đã tải ({current_video_to_load}) không khớp với video đang chọn ({browse_video}). Vui lòng 'Load Video Player' lại.")
                else:
                    with st.spinner(f"Loading keyframes for {current_video_to_load}..."):
                        # Đây là logic chậm, chỉ chạy khi nhấn nút này
                        prefix = f"{browse_kf}/keyframes/{current_video_to_load}"
                        paths = sorted([
                            p for p in system._get_structure_paths()
                            if p.startswith(prefix)
                        ])
                        st.session_state.browse_results = [{"path": p, "score": 1.0} for p in paths]
                        st.rerun() # Chạy lại để hiển thị keyframes
        # === KẾT THÚC SỬA ===

        st.markdown("---")
        
        if st.session_state.get('current_browse_video'):
            video_id = st.session_state.current_browse_video
            
            st.markdown(f"### 🎬 Video Player: {video_id}")
            youtube_url = system.youtube_url_map.get(video_id)
            if youtube_url:
                st.video(youtube_url)
            else:
                st.warning("YouTube URL not found")
            
            st.markdown("### ⏱️ Time Converter")
            c1, c2 = st.columns(2)
            
            with c1:
                st.markdown("**Time → Frame**")
                with st.form(key="time_to_frame"):
                    time_input = st.text_input("Time (1m25s or 1:25 or 85.5):", key="time_in")
                    if st.form_submit_button("Calculate"):
                        try:
                            total_sec = None
                            if 'm' in time_input:
                                parts = time_input.replace('s', '').split('m')
                                total_sec = int(parts[0]) * 60 + float(parts[1] if parts[1] else 0)
                            elif ':' in time_input:
                                parts = time_input.split(':')
                                total_sec = int(parts[0]) * 60 + float(parts[1])
                            else:
                                total_sec = float(time_input)
                            
                            if total_sec is not None:
                                fps = system.video_fps_map.get(video_id, 30.0)
                                if fps > 0:
                                    st.success(f"Frame: {round(total_sec * fps)}")
                        except:
                            st.error("Invalid format")
            
            with c2:
                st.markdown("**Frame → Time**")
                with st.form(key="frame_to_time"):
                    frame_input = st.text_input("Frame Index:", key="frame_in")
                    if st.form_submit_button("Calculate"):
                        try:
                            frame_idx = int(frame_input)
                            fps = system.video_fps_map.get(video_id, 30.0)
                            if fps > 0:
                                seconds = frame_idx / fps
                                st.success(f"Time: {format_time(seconds)}")
                        except:
                            st.error("Invalid frame index")
            
            st.markdown("---")
            st.markdown("### 🖼️ Keyframes")
            if st.session_state.get('browse_results'):
                display_gallery_with_video(
                    st.session_state.browse_results,
                    system,
                    key_prefix="browse",
                    cols_per_row=st.session_state.cols_per_row,
                    show_video_player=False, # Optimization
                )
            else:
                # MỚI: Thêm thông báo nếu chưa tải keyframes
                st.caption("Nhấn 'Load Keyframes' (ở trên) để xem thư viện ảnh.")

    # TÍNH NĂNG MỚI: TAB 6 - TRAKE SEARCH
    with tab6:
        st.markdown("### ⛓️ Trake (Sequential Event) Search")
        st.info("Tìm kiếm các sự kiện xảy ra theo đúng thứ tự thời gian (E1 -> E2 -> E3 -> E4).")
        
        if not milvus_collection or not model:
            st.error("Trake Search requires Milvus and Perception Model.")
        else:
            with st.form(key="trake_search_form"):
                st.markdown("#### Giai đoạn 1: Nhập các sự kiện (Macro UI)")
                e1_query = st.text_input(
                    "Sự kiện 1 (E1):",
                    placeholder="e.g., Cảnh sát vẫy tay đưa nghi phạm vào"
                )
                e2_query = st.text_input(
                    "Sự kiện 2 (E2) (Tùy chọn):",
                    placeholder="e.g., Cảnh sát chỉ tay vào vị trí"
                )
                e3_query = st.text_input(
                    "Sự kiện 3 (E3) (Tùy chọn):",
                    placeholder="e.g., Cảnh sát khom người xuống"
                )
                
                e4_query = st.text_input(
                    "Sự kiện 4 (E4) (Tùy chọn):",
                    placeholder="Sự kiện cuối cùng (nếu có)"
                )
                
                st.markdown("#### Giai đoạn 2: Cấu hình tìm kiếm (Macro Backend)")
                c1, c2 = st.columns(2)
                with c1:
                    trake_search_k = st.slider(
                        "Số ứng viên cho mỗi sự kiện (K)", 10, 500, 100,
                        help="Tìm K kết quả tốt nhất cho E1, E2, E3 trước khi lọc thứ tự."
                    )
                with c2:
                    max_sequences = st.slider(
                        "Số chuỗi kết quả tối đa", 1, 50, 10,
                        help="Số lượng chuỗi E1->E2->E3 tối đa sẽ hiển thị."
                    )
                
                submitted_trake = st.form_submit_button("🔍 Search Sequences", type="primary")

            if submitted_trake:
                queries = [q for q in [e1_query, e2_query, e3_query, e4_query] if q.strip()]
                
                kf_filter = st.session_state.kf_filter
                vid_filter = st.session_state.vid_filter
                
                if len(queries) < 1:
                    st.warning("Vui lòng nhập ít nhất 1 sự kiện (E1).")
                else:
                    # MỚI: Logic dịch
                    translated_queries = []
                    if st.session_state.translate_to_en:
                        with st.spinner("Đang dịch các sự kiện Trake..."):
                            start_translate = time.time()
                            for q in queries:
                                translated_q = translate_to_english(q)
                                translated_queries.append(translated_q)
                            end_translate = time.time()
                            st.info(f"Đã dịch {len(translated_queries)} câu trong {end_translate - start_translate:.2f}s: {translated_queries}")
                    else:
                        translated_queries = queries # Dùng bản gốc
                
                    with st.spinner(f"Đang tìm kiếm {len(translated_queries)} sự kiện..."):
                        all_results_list = []
                        search_start_time = time.time()
                        
                        for i, query in enumerate(translated_queries): # Dùng text đã dịch
                            st.write(f"Đang tìm E{i+1}: '{query}'...")
                            query_vector = encode_text_query(query, model, tokenizer, device)
                            event_results = search_milvus_for_trake(
                                milvus_collection, query_vector, trake_search_k, system
                            )
                            if not event_results:
                                st.error(f"Không tìm thấy kết quả nào cho E{i+1}. Dừng tìm kiếm.")
                                all_results_list = [] # Xóa kết quả
                                break
                            all_results_list.append(event_results)
                    
                    if all_results_list:
                        st.write("Đã tìm thấy các ứng viên. Đang lọc thứ tự thời gian...")
                        sequences = find_temporal_sequences(
                            all_results_list, 
                            max_sequences, 
                            kf_filter, 
                            vid_filter
                        )
                        end_time = time.time()
                        
                        st.success(f"✅ Đã tìm thấy {len(sequences)} chuỗi hợp lệ trong {end_time - search_start_time:.2f}s.")
                        st.session_state.trake_sequences_results = sequences
            
            st.markdown("---")
            st.markdown("#### Giai đoạn 3: Phân tích kết quả (Manual Micro-Analysis)")
            
            if not st.session_state.trake_sequences_results:
                st.info("Chưa có kết quả chuỗi. Vui lòng chạy tìm kiếm ở trên.")
            else:
                st.write(f"Hiển thị {len(st.session_state.trake_sequences_results)} chuỗi kết quả:")
                
                # This is the main loop for each sequence
                for i, seq in enumerate(st.session_state.trake_sequences_results):
                    st.subheader(f"Chuỗi kết quả #{i+1}")
                    
                    total_score = sum(res['score'] for res in seq)
                    st.caption(f"Video: {seq[0]['info']['folder_name']} | Tổng điểm: {total_score:.4f}")
                    
                    # === MODIFICATION START ===
                    # 1. Tạo các cột cho chuỗi này
                    #    Số cột sẽ bằng số sự kiện trong chuỗi (ví dụ: 1, 2, 3, 4)
                    num_events = len(seq)
                    cols = st.columns(num_events) 
                    
                    # 2. Lặp qua từng sự kiện và gán vào từng cột
                    for j, res in enumerate(seq):
                        with cols[j]:
                            st.markdown(f"**Sự kiện E{j+1}** (Score: {res['score']:.4f})")
                            
                            # 3. Hiển thị thủ công từng ảnh và thông tin
                            #    (Đây là logic rút gọn từ display_gallery_with_video)
                            rel_path = res["path"]
                            full_path = system.get_full_path(rel_path)
                            
                            if full_path and full_path.exists():
                                try:
                                    # Tải ảnh
                                    img = Image.open(str(full_path)).convert("RGB")
                                    buf = io.BytesIO()
                                    img.save(buf, format="JPEG")
                                    byte_im = buf.getvalue()

                                    # Lấy metadata
                                    img_info = res['info'] # Info đã được đính kèm
                                    video_id = img_info['folder_name']
                                    frame_name = img_info['image_name']
                                    frame_id = f"{video_id}/{frame_name}"
                                    
                                    # MỚI: Thêm 2 giá trị copy
                                    format_index_str, format_time_str, base_video_url, start_time_sec, time_ms, frame_idx, copy_idx_str, copy_time_str, fps = system.calculate_metadata(
                                        video_id, f"{frame_name}.jpg"
                                    )

                                    # Hiển thị
                                    st.image(byte_im, use_container_width=True)
                                    
                                    # === SỬA: Hiển thị Idx/Time với st.code ===
                                    st.markdown(f"**Idx:**")
                                    st.code(copy_idx_str, language=None)
                                    
                                    st.markdown(f"**Time (ms):**")
                                    st.code(copy_time_str, language=None)
                                    
                                    # MỚI: Thêm FPS
                                    st.caption(f"FPS: {fps:.2f}")
                                    # === KẾT THÚC SỬA ===
                                    
                                    # Nút Select
                                    if st.button("Select", key=f"trake_{i}_{j}_btn"):
                                        st.session_state.selected_frame_id = frame_id
                                        st.rerun()

                                    # Trình phát video
                                    if base_video_url and start_time_sec is not None:
                                        with st.expander("▶️ Watch Video"):
                                            st.video(base_video_url, start_time=start_time_sec)

                                except Exception as e:
                                    st.error(f"Error displaying trake item: {e}")
                            else:
                                st.warning(f"Image path not found: {rel_path}")

                    # === MODIFICATION END ===
                    st.markdown("---") # Phân cách giữa các chuỗi


if __name__ == "__main__":
    if not PERCEPTION_AVAILABLE:
        st.error("❌ Perception Models library missing")
    # MỚI: Kiểm tra thư viện dịch
    elif not TRANSLATE_AVAILABLE:
        st.error("❌ Thư viện 'googletrans' bị thiếu. Vui lòng cài đặt: pip install googletrans==4.0.0-rc1")
    else:
        main_ui()

Overwriting app.py


In [4]:
!pip uninstall -y scikit-learn

Found existing installation: scikit-learn 1.2.2
Uninstalling scikit-learn-1.2.2:
  Successfully uninstalled scikit-learn-1.2.2


In [ ]:
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient # <-- 1. Thêm thư viện này

# Lấy token từ secret
user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("NGROK_AUTHTOKEN")

# Xác thực với ngrok
ngrok.set_auth_token(ngrok_token) # <-- 2. Dùng token để xác thực

# Thiết lập cổng và kết nối
port = 8501
public_url = ngrok.connect(port) # <-- 3. Bây giờ dòng này sẽ chạy được

print(f"URL công khai của Streamlit: {public_url}")

# Chạy server Streamlit
!streamlit run app.py --server.port {port} --server.enableCORS=false --server.enableXsrfProtection=false

URL công khai của Streamlit: NgrokTunnel: "https://gelatinous-kanesha-unsapiently.ngrok-free.dev" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://35.238.232.11:8501

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute wi